In [ ]:
# === MODULE: src/namethatllm/config.py ===
TRAINING_CONFIG = {'version': '2.0.0', 'name': 'champion_two_stage_lora_14b', 'description': 'Champion Two-Stage Multi-Layer LoRA 14B Trainer (Stage 1 Pool Pre-training on 90k Passages + Stage 2 Contrastive Fine-Tuning with Family Margin Loss)', 'backbone': 'Qwen/Qwen3-14B-Base', 'keep_layers': 27, 'layers': [2, 4, 6, 9, 13, 17, 22, 27], 'max_len': 512, 'end_id': 151643, 'lora_r': 32, 'lora_alpha': 64, 'lora_dropout': 0.05, 'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], 'family_margin': 0.35, 'supcon_weight': 0.2, 'stage1': {'enabled': True, 'lr_lora': 0.00015, 'lr_head': 0.0008, 'batch_size': 2, 'gradient_accumulation_steps': 8, 'epochs': 1, 'max_samples': 90000, 'warmup_steps': 100}, 'stage2': {'enabled': True, 'lr_lora': 3e-05, 'lr_head': 0.0002, 'batch_size': 2, 'gradient_accumulation_steps': 4, 'epochs': 2, 'centered_weight': 0.6, 'warmup_steps': 50}, 'weight_decay': 0.01, 'output_dir': 'export_champion_qwen3_14b'}

"""
Canonical Configuration, Constants & Path Resolution
====================================================
Master canonical label ordering, provider families, simplex helpers,
and Kaggle filesystem asset discovery.
"""

import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
import numpy as np

# Canonical 18-Class Labels (Strict order from NTL two-stage LoRA part two)
LABELS = [
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3"
]
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}

FAMILIES = {
    "gpt": [0, 1, 2, 3, 4],
    "gemini": [5, 6, 7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
}

PRIOR_SCALES = {
    "qwen3_14b_p2": 1.00,
    "qwen3_14b_inhouse": 1.00,
    "qwen25_14b_p2": 0.85,
    "qwen25_14b_inhouse": 0.85,
    "deberta_p2": 0.80,
    "deberta_inhouse": 0.80,
    "qwen3_14b_p1": 0.50,
    "instyle": 0.45,
    "qwen3_8b_claude": 0.30,
    "qwen3_8b_inhouse": 0.30,
}

def align_cols(arr: np.ndarray, src_labels: list) -> np.ndarray:
    """Aligns columns of array to canonical 18-class order."""
    src_list = [x.decode('utf-8') if isinstance(x, (bytes, np.bytes_)) else str(x) for x in src_labels]
    idx = [src_list.index(l) for l in LABELS]
    if arr.ndim == 1:
        return arr[idx]
    elif arr.ndim == 2:
        if arr.shape[0] == len(src_list) and arr.shape[1] != len(src_list):
            return arr[idx, :]
        elif arr.shape[1] == len(src_list):
            return arr[:, idx]
    return arr[..., idx]

def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable log-softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    return x - c - np.log(np.sum(np.exp(x - c), axis=axis, keepdims=True) + 1e-15)

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - c)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-15)

def resolve_dataset_paths(input_dir: Optional[Path] = None) -> Tuple[Path, Path, Optional[Path]]:
    """Resolves train.csv, test.csv, and pool_v4.parquet across local and Kaggle environments."""
    if input_dir is None:
        input_dir = Path('/kaggle/input')
        if not input_dir.exists():
            input_dir = Path('.')

    candidates = [
        input_dir / 'competitions' / 'name-that-llm',
        input_dir / 'name-that-llm',
        Path('local_data'),
        Path('.')
    ]
    train_path = None
    test_path = None
    for c in candidates:
        if (c / 'train.csv').exists() and train_path is None:
            train_path = c / 'train.csv'
        if (c / 'test.csv').exists() and test_path is None:
            test_path = c / 'test.csv'

    if train_path is None:
        train_path = Path('local_data') / 'train.csv'
    if test_path is None:
        test_path = Path('local_data') / 'test.csv'

    pool_files = resolve_all_pool_files(input_dir)
    pool_file = pool_files[0] if pool_files else None

    return train_path, test_path, pool_file

def resolve_all_pool_files(input_dir: Optional[Path] = None) -> List[Path]:
    """Discovers all pretraining/unlabeled pool parquet files in priority order."""
    if input_dir is None:
        input_dir = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
    pool_files: List[Path] = []
    pool_patterns = ['nlm_pool_v5.parquet', 'pool_v4.parquet', 'nlm_matched_clean.parquet', 'nlm_pool_v2.parquet']
    for pat in pool_patterns:
        for p in input_dir.rglob(pat):
            if p not in pool_files:
                pool_files.append(p)
    return pool_files

def find_base_model(input_dir: Path, model_family: str, hidden_sz: int, num_layers: int) -> Optional[Path]:
    """Discovers base model directories from Hugging Face config.json inspection."""
    for cfg_p in input_dir.rglob('config.json'):
        try:
            cfg = json.load(open(cfg_p))
            m_type = str(cfg.get('model_type', '')).lower()
            h_dim = cfg.get('hidden_size', 0)
            n_lay = cfg.get('num_hidden_layers', 0)
            if model_family.lower() in m_type and h_dim == hidden_sz and n_lay == num_layers:
                return cfg_p.parent
        except Exception:
            continue
    return None

def _inspect_checkpoint_dim_and_keep(ckpt_dir: Path) -> Tuple[int, int, str]:
    """Returns (head_feature_dim, keep_layers, backbone_name) for checkpoint directory without full model loading."""
    dim = 0
    keep = 0
    backbone = ""
    meta_p = ckpt_dir / "meta.json"
    if meta_p.exists():
        try:
            m = json.load(open(meta_p))
            cfg = m.get("cfg", m)
            keep = int(cfg.get("keep", 0))
            backbone = str(m.get("backbone", cfg.get("backbone", ""))).lower()
        except Exception:
            pass

    heads_p = ckpt_dir / "heads.safetensors"
    if heads_p.exists():
        try:
            from safetensors import safe_open
            with safe_open(str(heads_p), framework="pt", device="cpu") as f:
                for k in f.keys():
                    if "plain.weight" in k or "weight" in k:
                        shape = f.get_tensor(k).shape
                        dim = shape[1] if len(shape) == 2 and shape[0] == 18 else shape[0]
                        break
        except Exception:
            pass
    elif (ckpt_dir / "heads.pt").exists():
        try:
            import torch
            sd = torch.load(ckpt_dir / "heads.pt", map_location="cpu")
            for k, v in sd.items():
                if hasattr(v, "shape") and len(v.shape) == 2:
                    dim = v.shape[1] if v.shape[0] == 18 else v.shape[0]
                    break
        except Exception:
            pass
    return dim, keep, backbone

def find_checkpoints(input_dir: Path) -> Dict[str, Path]:
    """Finds all fine-tuned LoRA and head checkpoints across safetensors and PyTorch formats."""
    ckpts = {}

    # 1. Check for heads.safetensors and heads.pt files
    head_candidates = list(input_dir.rglob('heads.safetensors')) + list(input_dir.rglob('heads.pt'))
    for p in head_candidates:
        p_str = str(p).lower().replace('\\', '/')
        is_fold = ('_f0' in p_str or '_f1' in p_str or '_f2' in p_str or '_f3' in p_str or '_f4' in p_str)

        if 'lora_qwen3_14b' in p_str or 'qwen3_14b_part2' in p_str or ('14b' in p_str and 'part2' in p_str and 'qwen3' in p_str):
            ckpts['qwen3_14b_p2'] = p.parent
        elif 'lora_qwen25_14b' in p_str or 'qwen25_14b_part2' in p_str or ('14b' in p_str and 'part2' in p_str and 'qwen2' in p_str):
            ckpts['qwen25_14b_p2'] = p.parent
        elif 'q14v5' in p_str:
            if 'qwen3_14b_v5' not in ckpts or (not is_fold):
                ckpts['qwen3_14b_v5'] = p.parent
        elif 'ts_q3b14' in p_str:
            if 'qwen3_14b_ts' not in ckpts or (not is_fold):
                ckpts['qwen3_14b_ts'] = p.parent
        elif 'lora14b_v1' in p_str:
            ckpts['qwen3_14b_p1'] = p.parent
        elif any(token in p_str for token in ('inhouse', 'ntl-inhouse', 'export_lora24', 'export_champion', 'lora-peft-training', 'peft_training')):
            dim, keep, backbone = _inspect_checkpoint_dim_and_keep(p.parent)
            if '2.5' in backbone or 'qwen25' in backbone or (dim == 40960 and (keep >= 30 or '25' in p_str or 'qwen25' in p_str)):
                ckpts['qwen25_14b_inhouse'] = p.parent
            elif ('qwen3' in backbone and '14b' in backbone) or (dim == 40960 or '14b' in p_str):
                ckpts['qwen3_14b_inhouse'] = p.parent
            elif 'deberta' in backbone or (dim == 8192 or 'deberta' in p_str):
                ckpts['deberta_inhouse'] = p.parent
            else:
                ckpts['qwen3_8b_inhouse'] = p.parent
        elif 'q8v5' in p_str:
            if 'qwen3_8b_v5' not in ckpts or (not is_fold):
                ckpts['qwen3_8b_v5'] = p.parent
        elif 'ts_q3b8' in p_str or 'ts8_s7' in p_str or 'l24x_q3b8' in p_str or 'll8v5' in p_str:
            if 'qwen3_8b_ts' not in ckpts or (not is_fold):
                ckpts['qwen3_8b_ts'] = p.parent
        elif 'lora8b_v1' in p_str:
            ckpts['qwen3_8b_p1'] = p.parent
        elif 'ts_q25_7b' in p_str or 'l24x_q25_7b' in p_str:
            if 'qwen25_7b_ts' not in ckpts or (not is_fold):
                ckpts['qwen25_7b_ts'] = p.parent
        elif 'instyle' in p_str:
            ckpts['instyle'] = p.parent

    # 2. Check for Claude specialist head
    for p in input_dir.rglob('claude_head.safetensors'):
        ckpts['qwen3_8b_claude'] = p.parent
        break
    if 'qwen3_8b_claude' not in ckpts:
        for p in input_dir.rglob('readouts.npz'):
            if 'claude' in str(p).lower():
                ckpts['qwen3_8b_claude'] = p.parent
                break

    # 3. Check for DeBERTa sequence classification head
    for p in list(input_dir.rglob('head.safetensors')) + list(input_dir.rglob('heads.safetensors')):
        if 'deberta' in str(p).lower():
            ckpts['deberta_p2'] = p.parent
            break

    # 4. Fallback search for instyle
    if 'instyle' not in ckpts:
        for p in input_dir.rglob('*'):
            p_s = str(p).lower().replace('\\', '/')
            if 'instyle' in p_s and ((p / 'adapter_model.safetensors').exists() or (p / 'adapter' / 'adapter_model.safetensors').exists() or (p / 'heads.safetensors').exists() or (p / 'heads.pt').exists()):
                ckpts['instyle'] = p
                break

    # 5. Fallback search for q14v5 / ts_q3b14 adapter
    if 'qwen3_14b_ts' not in ckpts:
        for p in input_dir.rglob('adapter_model.safetensors'):
            p_s = str(p).lower().replace('\\', '/')
            if 'q14v5' in p_s or 'ts_q3b14' in p_s:
                ckpts['qwen3_14b_ts'] = p.parent
                break

    return ckpts

def validate_production_assets(
    input_dir: Path,
    train_path: Path,
    test_path: Path,
    pool_files: List[Path],
    ckpts: Dict[str, Path],
    cfg: Optional[Dict[str, Any]] = None,
) -> Dict[str, Path]:
    """
    Fail-closed pre-flight validation for production inference runs (when NTL_SMOKE != '1').
    Raises FileNotFoundError immediately at startup if any required dataset, pool parquet,
    base model trunk, or checkpoint is missing.
    Returns a dict of verified base_model paths keyed by name.
    """
    if cfg is None:
        cfg = {}
    req = cfg.get("required_assets", {})
    missing: List[str] = []

    # 1. Verify competition CSVs
    if not train_path or not Path(train_path).exists():
        missing.append(f"Competition train.csv not found at: {train_path}")
    if not test_path or not Path(test_path).exists():
        missing.append(f"Competition test.csv not found at: {test_path}")

    # 2. Verify public pool parquet(s)
    require_pool = req.get("require_pool_parquet", True)
    valid_pools = [p for p in pool_files if p is not None and Path(p).exists()]
    if require_pool and len(valid_pools) == 0:
        missing.append(f"No public pool parquet (pool_v4.parquet / nlm_pool_v5.parquet) found under {input_dir}")

    # 3. Verify all required base model trunks
    default_bases = [
        {"name": "qwen3_14b", "family": "qwen3", "hidden_size": 5120, "num_layers": 40},
        {"name": "qwen25_14b", "family": "qwen2", "hidden_size": 5120, "num_layers": 48},
        {"name": "qwen3_8b", "family": "qwen3", "hidden_size": 4096, "num_layers": 36},
        {"name": "deberta_v3_large", "family": "deberta", "hidden_size": 1024, "num_layers": 24},
    ]
    base_specs = req.get("required_base_models", default_bases)
    verified_bases: Dict[str, Path] = {}
    for b_spec in base_specs:
        b_path = find_base_model(
            input_dir,
            b_spec["family"],
            int(b_spec["hidden_size"]),
            int(b_spec["num_layers"]),
        )
        if b_path is None or not Path(b_path).exists():
            missing.append(
                f"Required base model '{b_spec['name']}' (family={b_spec['family']}, "
                f"hidden={b_spec['hidden_size']}, layers={b_spec['num_layers']}) not found under {input_dir}"
            )
        else:
            verified_bases[b_spec["name"]] = b_path

    # 4. Verify all required fine-tuned checkpoints
    default_req_ckpts = [
        "qwen3_14b_p2", "qwen3_14b_p1", "qwen25_14b_p2",
        "deberta_p2", "qwen3_8b_claude"
    ]
    req_ckpts = req.get("required_checkpoints", default_req_ckpts)
    for ck_name in req_ckpts:
        if ck_name not in ckpts or not Path(ckpts[ck_name]).exists():
            missing.append(f"Required checkpoint '{ck_name}' not found under {input_dir} (discovered: {list(ckpts.keys())})")

    if missing:
        error_msg = (
            "[PRE-FLIGHT ASSET AUDIT FAILED] Missing required inference assets:\n  - "
            + "\n  - ".join(missing)
        )
        raise FileNotFoundError(error_msg)

    print(
        f"[Pre-Flight Audit PASSED] Verified {len(valid_pools)} pool file(s), "
        f"{len(verified_bases)} base trunks, and {len(ckpts)} checkpoints: {list(ckpts.keys())}"
    )
    return verified_bases


In [ ]:
# === MODULE: src/namethatllm/wpca_gaussian.py ===
"""
Full-Rank wPCA & ExactAuthorPosterior (2^18 Bitmask DP)
======================================================
Two-way variance decomposition, Ledoit-Wolf shrinkage floors,
crash-proof cluster splitting (max_size <= 14), and exact forward-backward
bitmask DP author posteriors under the unique-authorship constraint.
"""

from functools import lru_cache
from typing import Tuple, List, Dict, Optional, Any
import numpy as np
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.special import logsumexp
from sklearn.covariance import ledoit_wolf_shrinkage


def _two_way(Z: np.ndarray, y: np.ndarray, pinv: np.ndarray, G: int, K_classes: int = 18, iters: int = 4) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Iterative two-way ANOVA decomposition separating author means and prompt random effects."""
    N, d = Z.shape
    mu = np.zeros((K_classes, d), dtype=np.float64)
    P = np.zeros((G, d), dtype=np.float64)
    ng = np.bincount(pinv, minlength=G).clip(min=1)
    for _ in range(iters):
        R = Z - P[pinv]
        for k in range(K_classes):
            idx = (y == k)
            if idx.any():
                mu[k] = R[idx].mean(axis=0)
        R = Z - mu[y]
        P[:] = 0.0
        np.add.at(P, pinv, R)
        P /= ng[:, None]
    return mu, P, ng

def _shrunk(S: np.ndarray, alpha: float) -> np.ndarray:
    """Shrinks covariance matrix S towards spherical target."""
    d = S.shape[0]
    tr = np.trace(S) / d
    return (1.0 - alpha) * S + alpha * tr * np.eye(d)

def fit_shared_gaussian_lowram(X_tr_f32: np.ndarray, y: np.ndarray, prompts: np.ndarray,
                               d_target: int = 256, se_floor: float = 0.10) -> dict:
    """
    Fits shared Gaussian whitener with within-prompt centering and Ledoit-Wolf shrinkage floor.
    Returns unsupervised top-64 within-prompt principal components V_top64 (zero label leakage!).
    """
    N, D = X_tr_f32.shape
    pu, pinv = np.unique(prompts, return_inverse=True)
    G = len(pu)

    mx = X_tr_f32.mean(axis=0, dtype=np.float64).astype(np.float32)
    sx = X_tr_f32.std(axis=0, dtype=np.float64).astype(np.float32)
    sx = np.where(sx > 1e-6, sx, 1.0).astype(np.float32)

    # Standardize and subtract prompt means in-place (Within-Prompt Whitened PCA!)
    R = (X_tr_f32 - mx) / sx
    Pm = np.zeros((G, D), dtype=np.float32)
    np.add.at(Pm, pinv, R)
    Pm /= np.bincount(pinv, minlength=G)[:, None].clip(min=1).astype(np.float32)
    R -= Pm[pinv]
    del Pm

    d = int(min(d_target, max(N - G - 1, 1), D))
    Gram = (R @ R.T).astype(np.float64)
    lam_r, U = np.linalg.eigh(Gram)
    idx_top = np.argsort(lam_r)[::-1][:d]
    lam_top = np.clip(lam_r[idx_top], 1e-10, None)
    U_f32 = U[:, idx_top].astype(np.float32)
    V_f32 = (R.T @ U_f32) / np.sqrt(lam_top).astype(np.float32)[None, :]
    del R, Gram, U, U_f32

    # Unsupervised within-prompt top-64 principal components (zero label leakage)
    V_top64 = V_f32[:, :min(64, d)].copy()

    Z = (((X_tr_f32 - mx) / sx) @ V_f32).astype(np.float64)

    mu, P, ng = _two_way(Z, y, pinv, G, K_classes=K, iters=4)
    E = Z - mu[y] - P[pinv]
    dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = (E.T @ E) / dof
    ae = max(float(ledoit_wolf_shrinkage(E, assume_centered=True)), se_floor)
    Se = _shrunk(Se_raw, ae)

    Pc = P - P.mean(0)
    Sp_raw = (Pc.T @ Pc) / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = float(ledoit_wolf_shrinkage(Pc, assume_centered=True))
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se))
    Sp = (1.0 - ap) * Sp_raw + ap * (tr / d) * np.eye(d)

    L = np.linalg.cholesky(Se)
    Li = np.linalg.inv(L)
    M = 0.5 * ((Li @ Sp @ Li.T) + (Li @ Sp @ Li.T).T)
    lam, Q = np.linalg.eigh(M)
    lam = np.clip(lam, 1e-4, None)
    A = Q.T @ Li
    W_f32 = (V_f32 @ A.T.astype(np.float32)).astype(np.float32)
    m_whitened = mu @ A.T

    return {
        "mx": mx, "sx": sx, "W": W_f32, "m": m_whitened, "lam": lam,
        "V_top64": V_top64, "T1": 2.40, "Tg": 2.35, "ae": ae
    }

def transform_lowram(X_f32: np.ndarray, prm: dict, batch_rows: int = 1000) -> np.ndarray:
    """Projects representations into whitened space."""
    N = len(X_f32)
    d = prm["W"].shape[1]
    U_out = np.empty((N, d), dtype=np.float64)
    mx, sx, W = prm["mx"], prm["sx"], prm["W"]
    for s in range(0, N, batch_rows):
        chunk = ((X_f32[s : s + batch_rows] - mx) / sx) @ W
        U_out[s : s + batch_rows] = chunk.astype(np.float64)
    return U_out

def split_oversized_clusters_dist(D: np.ndarray, groups: np.ndarray, max_cluster_size: int = 14) -> np.ndarray:
    """Recursively splits clusters with size > max_cluster_size to prevent bitmask DP overflow."""
    if len(groups) < 2:
        return groups.copy()
    new_groups = np.zeros(len(groups), dtype=np.int64)
    next_id = 1
    for g in np.unique(groups):
        rows = np.flatnonzero(groups == g)
        if len(rows) <= max_cluster_size:
            new_groups[rows] = next_id
            next_id += 1
        else:
            n_sub = int(np.ceil(len(rows) / max_cluster_size))
            D_sub = D[np.ix_(rows, rows)].copy()
            np.fill_diagonal(D_sub, 0.0)
            D_sub = np.clip(0.5 * (D_sub + D_sub.T), 0.0, None)
            d_condensed = squareform(D_sub, checks=False)
            sub_labels = fcluster(linkage(d_condensed, method='average'), t=n_sub, criterion='maxclust')
            for sg in np.unique(sub_labels):
                s_rows = rows[sub_labels == sg]
                for start in range(0, len(s_rows), max_cluster_size):
                    new_groups[s_rows[start : start + max_cluster_size]] = next_id
                    next_id += 1
    return new_groups

@lru_cache(maxsize=4)
def _bitmask_levels(k: int = 18):
    """Precomputes bitmasks partitioned by population count (number of set bits)."""
    masks = np.arange(1 << k, dtype=np.int64)
    counts = np.fromiter((int(x).bit_count() for x in masks), dtype=np.uint8, count=len(masks))
    return tuple(masks[counts == n] for n in range(k + 1))

class ExactAuthorPosterior:
    """Exact shared-prompt Gaussian posteriors under author uniqueness via 2^18 bitmask DP."""
    def __init__(self, m: np.ndarray, lam: np.ndarray):
        self.m = np.asarray(m, dtype=np.float64)
        self.lam = np.asarray(lam, dtype=np.float64)
        self.K, self.d = self.m.shape
        self.levels = _bitmask_levels(self.K)
        self._subset_means = {}

    def single(self, u: np.ndarray, temperature: float = 2.40) -> np.ndarray:
        diff = u[None, :] - self.m
        dist = np.sum((diff * diff) / (1.0 + self.lam)[None, :], axis=1)
        log_p = -0.5 * dist / max(float(temperature), 1e-4)
        return log_p - logsumexp(log_p)

    def group(self, U: np.ndarray, temperature: float = 2.35) -> np.ndarray:
        U = np.asarray(U, dtype=np.float64)
        n = len(U)
        if n <= 1 or n > self.K:
            return np.vstack([self.single(U[i], temperature=temperature) for i in range(n)])

        if n not in self._subset_means:
            if len(self._subset_means) >= 3:
                self._subset_means.clear()
            masks = self.levels[n]
            membership = ((masks[:, None] >> np.arange(self.K)) & 1).astype(np.float32)
            self._subset_means[n] = membership @ self.m.astype(np.float32)

        terminal = self.levels[n]
        gamma = self.lam / (1.0 + n * self.lam)
        residual = U.sum(axis=0, keepdims=True) - self._subset_means[n].astype(np.float64)
        correction = 0.5 * np.einsum("nd,d,nd->n", residual, gamma, residual) / max(float(temperature), 1e-4)
        correction -= correction.max()

        # Passage-specific unary log-likelihoods A[i, author]
        A = -0.5 * np.sum((U[:, None, :] - self.m[None, :, :]) ** 2, axis=2) / max(float(temperature), 1e-4)
        A -= A.max(axis=1, keepdims=True)

        forward = np.full(1 << self.K, -np.inf, dtype=np.float64)
        forward[0] = 0.0
        for count in range(1, n + 1):
            masks = self.levels[count]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) != 0]
                forward[chosen] = np.logaddexp(forward[chosen], forward[chosen ^ bit] + A[count - 1, author])

        backward = np.full_like(forward, -np.inf)
        backward[terminal] = correction
        for count in range(n - 1, -1, -1):
            masks = self.levels[count]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) == 0]
                backward[chosen] = np.logaddexp(backward[chosen], A[count, author] + backward[chosen | bit])

        result = np.empty((n, self.K), dtype=np.float64)
        for row in range(n):
            masks = self.levels[row]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) == 0]
                result[row, author] = logsumexp(forward[chosen] + A[row, author] + backward[chosen | bit]) - backward[0]

        result -= logsumexp(result, axis=1, keepdims=True)
        return result

def scores_exact_dp(U_test: np.ndarray, groups: np.ndarray, prm: dict, D_dist: np.ndarray) -> np.ndarray:
    """Wraps ExactAuthorPosterior with oversized cluster splitting guardrail."""
    N = len(U_test)
    if N == 0:
        return np.zeros((0, K), dtype=np.float64)
    if N == 1:
        solver = ExactAuthorPosterior(prm["m"], prm["lam"])
        return solver.single(U_test[0], temperature=prm.get("T1", 2.40))[None, :]

    safe_groups = split_oversized_clusters_dist(D_dist, groups, max_cluster_size=14)
    solver = ExactAuthorPosterior(prm["m"], prm["lam"])
    T1 = prm.get("T1", 2.40)
    Tg = prm.get("Tg", 2.35)

    lp_dp = np.zeros((N, K), dtype=np.float64)
    for g in np.unique(safe_groups):
        idx_g = np.where(safe_groups == g)[0]
        if len(idx_g) == 1:
            lp_dp[idx_g] = solver.single(U_test[idx_g[0]], temperature=T1)
        else:
            lp_dp[idx_g] = solver.group(U_test[idx_g], temperature=Tg)
    return lp_dp


In [ ]:
# === MODULE: src/namethatllm/neural_extractor.py ===
"""
Universal Neural Trunk Loader & Feature Extractor
=================================================
Manages multi-GPU sharding (2xT4 GPUs, max 13.5 GiB), layer truncation (keep_layers),
dynamic bfloat16 row-level retry for 14B fp16 overflow rescue, LayerNorm heads,
and bit-exact CPU fp16 snapshot/restore for shared trunks.
"""

import gc
import os
import re
import json
import time
import shutil
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel, AutoTokenizer
from safetensors.torch import load_file


def infer_adapter_max_layer(ckpt_dir: Path) -> Optional[int]:
    """Returns 1-indexed max transformer layer modified by the LoRA adapter (e.g. 24 or 27)."""
    ckpt_dir = Path(ckpt_dir)
    for rel in (
        "adapter_model.safetensors",
        "adapter/adapter_model.safetensors",
        "adapter_model.bin",
        "adapter/adapter_model.bin",
        "adapter_model.pt",
        "adapter/adapter_model.pt",
    ):
        p = ckpt_dir / rel
        if p.exists():
            try:
                if p.suffix == ".safetensors":
                    sd = load_file(str(p))
                else:
                    sd = torch.load(str(p), map_location="cpu", weights_only=False)
                    if hasattr(sd, "state_dict") and callable(sd.state_dict):
                        sd = sd.state_dict()
                layer_nums = []
                for k in sd.keys():
                    m = re.search(r"layers\.(\d+)\.", str(k))
                    if m:
                        layer_nums.append(int(m.group(1)) + 1)
                if layer_nums:
                    return max(layer_nums)
            except Exception:
                continue
    return None

def in_place_copy(dst: torch.Tensor, src: torch.Tensor):
    """In-place weight copy."""
    dst.data.copy_(src.to(device=dst.device, dtype=dst.dtype))

def _resolve_module_name(k: str, state_dict: dict) -> Optional[str]:
    """Resolves module name across various AutoModel and PEFT naming schemes."""
    raw = k.split(".lora_A")[0]
    candidates = [raw, raw.replace("base_model.model.", "")]
    if "layers." in raw:
        rel = raw[raw.index("layers."):]
        candidates.extend([rel, f"model.{rel}"])
    for c in candidates:
        if c in state_dict and hasattr(state_dict[c], "weight"):
            return c
    return None

def merge_lora(model: nn.Module, adapter_path: Path, sign: float = +1.0) -> bool:
    """In-place merges or unmerges (+1.0 or -1.0) a LoRA adapter directly into trunk weights."""
    weights_path = adapter_path / "adapter_model.safetensors"
    if not weights_path.exists():
        weights_path = adapter_path / "adapter" / "adapter_model.safetensors"
    if not weights_path.exists():
        return False

    sd = load_file(str(weights_path))
    cfg_p = adapter_path / "adapter_config.json"
    if not cfg_p.exists():
        cfg_p = adapter_path / "adapter" / "adapter_config.json"
    scale = 1.0
    if cfg_p.exists():
        c = json.load(open(cfg_p))
        scale = float(c.get("lora_alpha", 16)) / float(c.get("r", 16))

    state_dict = dict(model.named_modules())
    pairs = {}
    for k in sd.keys():
        if "lora_A" in k:
            b_key = k.replace("lora_A", "lora_B")
            if b_key in sd:
                m_name = _resolve_module_name(k, state_dict)
                if m_name is not None:
                    pairs[m_name] = (sd[k], sd[b_key])

    applied = 0
    with torch.no_grad():
        for m_name, (A, B) in pairs.items():
            if m_name in state_dict:
                mod = state_dict[m_name]
                if hasattr(mod, "weight"):
                    A_d = A.to(mod.weight.device, dtype=torch.float32)
                    B_d = B.to(mod.weight.device, dtype=torch.float32)
                    delta = (B_d @ A_d) * (scale * sign)
                    mod.weight.data.add_(delta.to(mod.weight.dtype))
                    applied += 1
    return applied > 0

@torch.no_grad()
def forward_and_pool_batch(model: nn.Module, enc: dict, layers: List[int], norms: nn.ModuleList,
                           dtype: torch.dtype = torch.float16, drop_first_token: bool = True):
    """
    Runs forward pass in `dtype` (float16 on T4).
    CRITICAL: If any target layer overflows fp16 (Inf/NaN), recomputes ONLY the overflowed
    rows in `torch.bfloat16` and pools directly in float32 before restoring `dtype`!
    """
    hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states

    B = enc["input_ids"].shape[0]
    first_norm_dev = norms[0].weight.device

    # Check if any selected layer has Inf/NaN per row
    bad_mask = torch.zeros(B, dtype=torch.bool, device="cpu")
    for l in layers:
        bad_mask |= ~torch.isfinite(hs[l]).all(dim=(1, 2)).cpu()

    n_bad = int(bad_mask.sum().item())
    m = enc["attention_mask"].unsqueeze(-1).float().to(first_norm_dev)
    if drop_first_token:
        m[:, 0, :] = 0.0
    den = m.sum(dim=1).clamp(min=1.0)

    # Pool finite rows first in float32
    pooled_layers = [torch.zeros((B, model.config.hidden_size), dtype=torch.float32, device=first_norm_dev) for _ in layers]
    good_rows = torch.where(~bad_mask)[0].to(first_norm_dev)
    if len(good_rows) > 0:
        for idx_l, l in enumerate(layers):
            h_good = hs[l][good_rows.to(hs[l].device)].to(first_norm_dev).float()
            pooled_layers[idx_l][good_rows] = (h_good * m[good_rows]).sum(dim=1) / den[good_rows]

    # Rescue overflowed rows in bfloat16 (exact exponent range as float32, NEVER overflows!)
    if n_bad > 0:
        bad_rows = torch.where(bad_mask)[0]
        del hs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        model.to(torch.bfloat16)
        enc_bad = {k: v[bad_rows] for k, v in enc.items()}
        hs_bad = model(**enc_bad, output_hidden_states=True, use_cache=False).hidden_states
        bad_rows_dev = bad_rows.to(first_norm_dev)
        for idx_l, l in enumerate(layers):
            h_b = hs_bad[l].to(first_norm_dev).float()
            h_b = torch.nan_to_num(h_b, nan=0.0, posinf=1e4, neginf=-1e4)
            pooled_layers[idx_l][bad_rows_dev] = (h_b * m[bad_rows_dev]).sum(dim=1) / den[bad_rows_dev]
        del hs_bad
        model.to(dtype)

    out = torch.cat([nm(pooled_layers[idx_l]) for idx_l, nm in enumerate(norms)], dim=-1)
    return out.cpu().numpy(), n_bad

def _flatten_state_dict(obj: Any, prefix: str = "", out: Optional[dict] = None) -> dict:
    if out is None:
        out = {}
    if hasattr(obj, "state_dict") and callable(obj.state_dict):
        obj = obj.state_dict()
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_k = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, (dict, list, tuple)) or (hasattr(v, "state_dict") and callable(v.state_dict)):
                _flatten_state_dict(v, new_k, out)
            else:
                out[new_k] = v
                if prefix in ("state_dict", "model", "heads", "head") and str(k) not in out:
                    out[str(k)] = v
    elif isinstance(obj, (list, tuple)):
        for idx, item in enumerate(obj):
            new_k = f"{prefix}.{idx}" if prefix else str(idx)
            _flatten_state_dict(item, new_k, out)
    return out

def load_head_file(path_or_dir: Path) -> dict:
    """Unified loader for classification head state dicts (.safetensors, .pt, .bin, .npz)."""
    p = Path(path_or_dir)
    res = {}
    if p.is_file():
        if p.suffix == ".safetensors":
            res = load_file(str(p))
        elif p.suffix in (".pt", ".bin"):
            raw = torch.load(str(p), map_location="cpu", weights_only=False)
            res = _flatten_state_dict(raw)
        elif p.suffix == ".npz":
            npz = np.load(str(p), allow_pickle=True)
            res = {k: npz[k] for k in npz.files}
        if res:
            print(f"[Head File] {p.name} loaded ({len(res)} keys): {list(res.keys())[:15]}")
        return res
    for fname in ("heads.safetensors", "head.safetensors", "heads.pt", "head.pt", "readouts.npz"):
        cand = p / fname
        if cand.exists():
            return load_head_file(cand)
    return {}

def resolve_head_tensors(heads_dict: dict) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
    """Resolves plain and centered head weights and biases matching American/British naming and handles transposed weights."""
    W_p, b_p, W_c, b_c = None, None, None, None
    for wk, bk in [
        ("plain.weight", "plain.bias"),
        ("plain_coef", "plain_intercept"),
        ("head.weight", "head.bias"),
        ("classifier.weight", "classifier.bias"),
        ("fc.weight", "fc.bias"),
        ("W_plain", "b_plain"),
        ("weight", "bias"),
        ("W", "b"),
    ]:
        if wk in heads_dict and bk in heads_dict:
            W_p, b_p = heads_dict[wk], heads_dict[bk]
            break
        elif wk in heads_dict and bk not in heads_dict:
            W_p = heads_dict[wk]
            b_p = np.zeros(K, dtype=np.float32)
            break

    for wk, bk in [
        ("centered.weight", "centered.bias"),
        ("centred.weight", "centred.bias"),
        ("center.weight", "center.bias"),
        ("twin.weight", "twin.bias"),
        ("center_coef", "center_intercept"),
        ("centred_coef", "centred_intercept"),
        ("W_center", "b_center"),
        ("W_twin", "b_twin"),
    ]:
        if wk in heads_dict and bk in heads_dict:
            W_c, b_c = heads_dict[wk], heads_dict[bk]
            break
        elif wk in heads_dict and bk not in heads_dict:
            W_c = heads_dict[wk]
            b_c = np.zeros(K, dtype=np.float32)
            break

    if W_p is not None and hasattr(W_p, "cpu"):
        W_p = W_p.cpu().numpy()
    if b_p is not None and hasattr(b_p, "cpu"):
        b_p = b_p.cpu().numpy()
    if W_c is not None and hasattr(W_c, "cpu"):
        W_c = W_c.cpu().numpy()
    if b_c is not None and hasattr(b_c, "cpu"):
        b_c = b_c.cpu().numpy()

    # Transpose Guard: if shape is (D, K) where K == 18, transpose to canonical (18, D)
    if W_p is not None and W_p.ndim == 2:
        if W_p.shape[1] == K and W_p.shape[0] != K:
            W_p = W_p.T
    if W_c is not None and W_c.ndim == 2:
        if W_c.shape[1] == K and W_c.shape[0] != K:
            W_c = W_c.T

    return W_p, b_p, W_c, b_c

def compute_centered_logits(Z_test: np.ndarray, groups: np.ndarray, W_c: np.ndarray, b_c: np.ndarray, src_labels: list) -> Tuple[np.ndarray, np.ndarray]:
    """Computes prompt-topic centered twin logits for valid multi-sample clusters."""
    N = len(Z_test)
    out_lp = np.zeros((N, K), dtype=np.float64)
    if N < 2 or W_c is None:
        return out_lp, np.zeros(N, dtype=bool)
    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_g = inv_g + 1
    sz = np.bincount(safe_g)[safe_g]
    valid = (sz >= 2) & (sz <= 18)
    if not valid.any():
        return out_lp, valid
    Zc = Z_test.copy()
    for g in np.unique(safe_g[valid]):
        mask = (safe_g == g)
        Zc[mask] -= Zc[mask].mean(axis=0, keepdims=True)
    raw_c = Zc[valid] @ W_c.T + b_c
    out_lp[valid] = log_softmax(align_cols(raw_c, src_labels))
    return out_lp, valid

class UniversalLoRATrunkLoader:
    def __init__(self, base_dir: Path, keep_layers: int, layers: List[int], max_len: int = 447):
        if base_dir is not None and not (base_dir / "config.json").exists():
            for sub in base_dir.rglob("config.json"):
                base_dir = sub.parent
                break
        self.base_dir = base_dir
        self.keep_layers = keep_layers
        self.layers = layers
        self.max_len = max_len
        self.total_rescued = 0
        self.tok = AutoTokenizer.from_pretrained(str(base_dir), trust_remote_code=True)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token or "<|endoftext|>"

        cfg = AutoConfig.from_pretrained(str(base_dir), trust_remote_code=True)
        cfg.num_hidden_layers = keep_layers
        if getattr(cfg, "layer_types", None) is not None:
            cfg.layer_types = cfg.layer_types[:keep_layers]
        cfg.use_cache = False

        n_gpus = torch.cuda.device_count()
        if n_gpus >= 2:
            max_memory = {0: "13.5GiB", 1: "13.5GiB"}
            self.model = AutoModel.from_pretrained(
                str(base_dir), config=cfg, device_map="auto", max_memory=max_memory,
                torch_dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True
            )
        elif n_gpus == 1:
            self.model = AutoModel.from_pretrained(
                str(base_dir), config=cfg, device_map="auto",
                torch_dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True
            )
        else:
            self.model = AutoModel.from_pretrained(str(base_dir), config=cfg, torch_dtype=torch.float32, low_cpu_mem_usage=True)

        # Disable final trunk norm so last layer is not double normalized
        if hasattr(self.model, "norm"):
            self.model.norm = nn.Identity()
        elif hasattr(self.model, "model") and hasattr(self.model.model, "norm"):
            self.model.model.norm = nn.Identity()

        self.model.eval()
        self.device_norms = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # nn.LayerNorm with both weight and bias matching heads.safetensors
        hidden_dim = self.model.config.hidden_size
        self.norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim)
            for _ in layers
        ]).to(self.device_norms).float()

        self.norms.eval()
        self.first_dev = next(self.model.parameters()).device

    def snapshot_weights(self, adapter_path: Path) -> dict:
        """Saves CPU fp16 copies of trunk modules modified by adapter."""
        weights_path = adapter_path / "adapter_model.safetensors"
        if not weights_path.exists():
            weights_path = adapter_path / "adapter" / "adapter_model.safetensors"
        if not weights_path.exists():
            return {}
        sd = load_file(str(weights_path))
        state_dict = dict(self.model.named_modules())
        backup = {}
        for k in sd.keys():
            if "lora_A" in k:
                m_name = _resolve_module_name(k, state_dict)
                if m_name is not None and m_name in state_dict and hasattr(state_dict[m_name], "weight"):
                    backup[m_name] = state_dict[m_name].weight.detach().cpu().half().clone()
        return backup

    def restore_weights(self, backup: dict):
        """Restores base trunk weights bit-for-bit without FP16 subtraction drift."""
        state_dict = dict(self.model.named_modules())
        with torch.no_grad():
            for m_name, orig in backup.items():
                if m_name in state_dict and hasattr(state_dict[m_name], "weight"):
                    state_dict[m_name].weight.data.copy_(orig.to(state_dict[m_name].weight.device))

    def set_active_layers(self, layers: List[int], max_len: Optional[int] = None):
        """Updates active pooled layers and instantiates LayerNorms matching the checkpoint."""
        self.layers = list(layers)
        if max_len is not None:
            self.max_len = max_len
        hidden_dim = self.model.config.hidden_size
        self.norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim)
            for _ in self.layers
        ]).to(self.device_norms).float()
        self.norms.eval()

    def load_norm_weights(self, heads_sd: dict):
        """Loads LayerNorm weights and biases from head state dict after resetting to identity."""
        with torch.no_grad():
            # First reset every ln in self.norms to identity so weights/biases from previous checkpoint never bleed
            for ln in self.norms:
                ln.weight.data.fill_(1.0)
                if hasattr(ln, "bias") and ln.bias is not None:
                    ln.bias.data.zero_()

            # Support all norm key prefixes: norms.{i}, layer_norms.{i}, ln.{i}, norm.{i}
            for i, ln in enumerate(self.norms):
                for pfx in ("norms", "layer_norms", "ln", "norm"):
                    w_k = f"{pfx}.{i}.weight"
                    b_k = f"{pfx}.{i}.bias"
                    if w_k in heads_sd:
                        ln.weight.copy_(heads_sd[w_k].to(ln.weight.device, ln.weight.dtype))
                    elif f"{pfx}.{i}" in heads_sd and hasattr(heads_sd[f"{pfx}.{i}"], "shape") and heads_sd[f"{pfx}.{i}"].shape == ln.weight.shape:
                        ln.weight.copy_(heads_sd[f"{pfx}.{i}"].to(ln.weight.device, ln.weight.dtype))

                    if b_k in heads_sd and hasattr(ln, "bias") and ln.bias is not None:
                        ln.bias.copy_(heads_sd[b_k].to(ln.bias.device, ln.bias.dtype))

            # Check for stacked 2D tensor: norms_weight or norms.weight of shape (len(self.layers), H)
            for stk in ("norms_weight", "norms.weight", "layer_norms.weight", "ln.weight"):
                if stk in heads_sd and hasattr(heads_sd[stk], "ndim") and heads_sd[stk].ndim == 2:
                    stk_t = heads_sd[stk]
                    if stk_t.shape[0] == len(self.norms):
                        for i, ln in enumerate(self.norms):
                            ln.weight.copy_(stk_t[i].to(ln.weight.device, ln.weight.dtype))

    def _extract_batch_ids(self, id_list: List[List[int]], batch_size: int = 8) -> np.ndarray:
        order = np.argsort([len(x) for x in id_list], kind="stable")
        total_dim = len(self.layers) * self.model.config.hidden_size
        Z = np.zeros((len(id_list), total_dim), dtype=np.float32)

        for s in range(0, len(order), batch_size):
            idx = order[s : s + batch_size]
            enc = self.tok.pad({"input_ids": [id_list[i] for i in idx]}, padding=True, return_tensors="pt")
            enc = {k: v.to(self.first_dev) for k, v in enc.items()}
            out_np, n_bad = forward_and_pool_batch(self.model, enc, self.layers, self.norms, torch.float16, drop_first_token=True)
            self.total_rescued += n_bad
            Z[idx] = out_np
        return Z

    @torch.no_grad()
    def extract_exact(self, texts: List[str], batch_size: int = 8, end_id: int = 151643) -> np.ndarray:
        t0 = time.time()
        self.total_rescued = 0
        raw_ids = self.tok(list(texts), add_special_tokens=False, truncation=True, max_length=self.max_len)["input_ids"]
        ids = [x + [end_id] for x in raw_ids]
        Z = self._extract_batch_ids(ids, batch_size=batch_size)
        print(f"[Extractor] Extracted {len(texts)} samples in {time.time() - t0:.2f}s (Total BF16 Rescued Rows: {self.total_rescued}).")
        return Z

    def free_memory(self):
        del self.model, self.norms
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def read_checkpoint_meta(ckpt_dir: Path, default_keep: int, default_layers: List[int], default_max_len: int = 447) -> dict:
    """Reads keep, layers, max_len, end_id, and labels dynamically from meta.json or infers from adapter weights."""
    ckpt_dir = Path(ckpt_dir)
    meta_p = ckpt_dir / "meta.json"
    labels = list(LABELS)
    keep = default_keep
    layers = list(default_layers)
    max_len = default_max_len
    end_id = 151643
    if meta_p.exists():
        try:
            m = json.load(open(meta_p))
            labels = m.get("labels", m.get("classes", labels))
            cfg = m.get("cfg", m)
            keep = int(cfg.get("keep", keep))
            layers = list(cfg.get("layers", layers))
            max_len = int(cfg.get("max_len", max_len))
            for ek in ("end_token_id", "eos_id", "eos_token_id", "end_token"):
                if ek in cfg and isinstance(cfg[ek], int):
                    end_id = int(cfg[ek])
                    break
        except Exception:
            pass
    else:
        # Infer exact layer depth from adapter weights (e.g. l24 vs l27)
        max_l = infer_adapter_max_layer(ckpt_dir)
        if max_l == 24:
            keep = 24
            layers = [2, 4, 6, 8, 12, 16, 20, 24]
        elif max_l == 27:
            keep = 27
            layers = [2, 4, 6, 9, 13, 17, 22, 27]
        elif max_l == 32:
            keep = 32
            layers = [3, 5, 8, 11, 16, 21, 27, 32]
        elif max_l is not None:
            keep = max_l
        else:
            dir_str = str(ckpt_dir).lower()
            if any(x in dir_str for x in ("l24", "q14v5", "ts_q3b14", "ts_q3b8", "q8v5", "l24x")):
                keep = 24
                layers = [2, 4, 6, 8, 12, 16, 20, 24]

    return {"keep": keep, "layers": layers, "max_len": max_len, "end_id": end_id, "labels": labels}

def extract_deberta_checkpoint(
    ckpt_dir: Path,
    base_deb_dir: Optional[Path],
    texts_test: List[str],
    texts_train: List[str],
    y_train: np.ndarray,
    max_len: int = 512,
    batch_size: int = 8
) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], list]:
    """
    Dedicated feature extractor for DeBERTa-v3-large sequence classification checkpoints.
    Probes drop_first_token and tokenization settings on train.csv to guarantee optimal accuracy.
    """
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    tok_dir = str(base_deb_dir if base_deb_dir else ckpt_dir)
    tok = AutoTokenizer.from_pretrained(tok_dir, use_fast=True)

    heads_sd = load_head_file(ckpt_dir)
    W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)

    meta = read_checkpoint_meta(ckpt_dir, default_keep=24, default_layers=[3, 6, 9, 12, 15, 18, 21, 24], default_max_len=max_len)
    ckpt_labels = meta["labels"]
    layers = meta["layers"]
    max_len = meta["max_len"]

    if (ckpt_dir / "encoder" / "config.json").exists():
        model_src = ckpt_dir / "encoder"
    elif (ckpt_dir / "config.json").exists() and ((ckpt_dir / "model.safetensors").exists() or (ckpt_dir / "pytorch_model.bin").exists()):
        model_src = ckpt_dir
    elif base_deb_dir and (base_deb_dir / "config.json").exists():
        model_src = base_deb_dir
    else:
        model_src = ckpt_dir

    stage_dir = Path("/kaggle/temp/deberta_p2_stage")
    staged = False
    load_src = model_src
    if Path("/kaggle/temp").exists() and os.access("/kaggle/temp", os.W_OK):
        try:
            if stage_dir.exists():
                shutil.rmtree(stage_dir, ignore_errors=True)
            shutil.copytree(model_src, stage_dir)
            load_src = stage_dir
            staged = True
        except Exception:
            load_src = model_src

    try:
        model = AutoModel.from_pretrained(str(load_src), torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to(dev)
    finally:
        if staged and stage_dir.exists():
            shutil.rmtree(stage_dir, ignore_errors=True)

    model.eval()

    hidden_dim = model.config.hidden_size
    if W_p is not None and W_p.shape[1] > 0:
        expected_layers = W_p.shape[1] // hidden_dim
        if len(layers) != expected_layers:
            norm_keys = [k for k in heads_sd if k.startswith("norms.") and k.endswith(".weight")]
            if len(norm_keys) == expected_layers:
                layers = layers[:expected_layers] if len(layers) > expected_layers else list(range(expected_layers))
            elif len(layers) > expected_layers:
                layers = layers[:expected_layers]

    norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in layers]).to(dev).float()
    with torch.no_grad():
        for i, ln in enumerate(norms):
            w_k = f"norms.{i}.weight"
            b_k = f"norms.{i}.bias"
            if w_k in heads_sd:
                ln.weight.copy_(heads_sd[w_k].to(dev, ln.weight.dtype))
            if b_k in heads_sd:
                ln.bias.copy_(heads_sd[b_k].to(dev, ln.bias.dtype))
    norms.eval()

    def pool_texts(txt_list: List[str], drop_first: bool = True) -> np.ndarray:
        if len(txt_list) == 0:
            return np.zeros((0, len(layers) * hidden_dim), dtype=np.float32)
        order = np.argsort([len(t) for t in txt_list], kind="stable")
        all_vecs = np.zeros((len(txt_list), len(layers) * hidden_dim), dtype=np.float32)
        for s in range(0, len(order), batch_size):
            idx = order[s : s + batch_size]
            chunk = [txt_list[i] for i in idx]
            enc = tok(chunk, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
            enc = {k: v.to(dev) for k, v in enc.items()}
            with torch.no_grad():
                hs = model(**enc, output_hidden_states=True).hidden_states
                m = enc["attention_mask"].unsqueeze(-1).float()
                if drop_first:
                    m[:, 0, :] = 0.0
                den = m.sum(dim=1).clamp(min=1.0)
                pooled = [((torch.nan_to_num(hs[l].float(), nan=0.0, posinf=1e4, neginf=-1e4) * m).sum(dim=1) / den) for l in layers]
                out = torch.cat([norms[i](pooled[i]) for i in range(len(layers))], dim=-1)
                all_vecs[idx] = out.cpu().numpy()
        return all_vecs

    # Probe drop_first_token on train.csv[:180]
    probe_n = min(180, len(texts_train))
    best_drop = True
    if probe_n > 0 and W_p is not None:
        best_acc = -1.0
        for cand_drop in [True, False]:
            z_pr = pool_texts(texts_train[:probe_n], drop_first=cand_drop)
            preds = align_cols(z_pr @ W_p.T + b_p, ckpt_labels).argmax(axis=1)
            acc = float((preds == y_train[:probe_n]).mean())
            if acc > best_acc:
                best_acc = acc
                best_drop = cand_drop
        print(f"[DeBERTa Extractor] Probed drop_first={best_drop} (accuracy = {best_acc:.4f})")

    Z_test = pool_texts(texts_test, drop_first=best_drop)
    Z_train = pool_texts(texts_train, drop_first=best_drop) if len(texts_train) > 0 else np.zeros((0, len(layers) * hidden_dim), dtype=np.float32)

    del model, norms
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return Z_test, Z_train, W_p, b_p, W_c, b_c, ckpt_labels


In [ ]:
# === MODULE: src/namethatllm/ablation.py ===
"""
Zero-Leakage Unseen Holdout Benchmark & Forward Stepwise Ablation Engine
========================================================================
Constructs a strictly unseen evaluation dataset from mounted parquet/CSV corpora
by subtracting all passages present in `train.csv` and `pool_v4.parquet`
(the training sets of the pre-trained Stage-1/Stage-2 checkpoints), quarantines
those holdout passages from `lexical_engine.py` and `trainer.py`, and evaluates:
1. Standalone (Solo) Unseen Accuracy of each model.
2. Forward Stepwise Cumulative Accuracy (1 model -> +2nd model -> ... -> Full Ensemble).
"""

import os
import gc
import hashlib
from itertools import combinations
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Set
import numpy as np
import pandas as pd
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score


def sinkhorn_balancing(P: np.ndarray, iters: int = 200, eps: float = 1e-9) -> np.ndarray:
    """Fast vectorized Sinkhorn mass balancing for balanced multiclass prediction."""
    P = np.clip(P, eps, 1.0)
    for _ in range(iters):
        P = P / P.sum(axis=1, keepdims=True).clip(min=eps)
        P = P / P.sum(axis=0, keepdims=True).clip(min=eps)
    return P / P.sum(axis=1, keepdims=True).clip(min=eps)

def _fast_sinkhorn_bacc(lp_mat: np.ndarray, y_true: np.ndarray, iters: int = 40) -> float:
    """Computes balanced accuracy after fast Sinkhorn mass equalization."""
    P = softmax(lp_mat, axis=1)
    if len(P) >= 18:
        P = sinkhorn_balancing(P, iters=iters)
    return float(balanced_accuracy_score(y_true, P.argmax(axis=1)))

def _text_hash(text: str) -> str:
    """Normalized SHA1 hash of whitespace-collapsed text to guarantee zero exact/near-exact leakage."""
    norm = " ".join(str(text).strip().split())
    return hashlib.sha1(norm.encode("utf-8", errors="ignore")).hexdigest()

def build_unseen_holdout_dataset(
    input_dir: Path,
    train_path: Optional[Path] = None,
    samples_per_class: int = 20,
    seed: int = 42,
) -> Tuple[pd.DataFrame, Set[str]]:
    """
    Builds a balanced 18-class unseen holdout dataset (`df_unseen`) by:
    1. Hashing all texts in `train.csv` and `pool_v4.parquet` (and `pool_v1.parquet`) as `contaminated_hashes`.
    2. Scanning `nlm_matched_clean.parquet`, `nlm_pool_v5.parquet`, and `openrouter-harvest`
       for passages whose text hash is strictly NOT in `contaminated_hashes`.
    3. Sampling up to `samples_per_class` passages per class (e.g., 18 * 20 = 360 unseen passages)
       and returning `(df_unseen, holdout_hashes)` so `lexical_engine.py` and `trainer.py`
       exclude `holdout_hashes` from training!
    """
    input_dir = Path(input_dir) if input_dir else Path(".")
    contaminated_hashes: Set[str] = set()

    # 1. Mark all train.csv rows as contaminated (seen by all Stage-2 LoRAs)
    if train_path and Path(train_path).exists():
        try:
            df_tr = pd.read_csv(train_path, usecols=["text"])
            contaminated_hashes.update(_text_hash(t) for t in df_tr["text"].fillna(""))
        except Exception:
            try:
                df_tr = pd.read_csv(train_path)
                if "text" in df_tr.columns:
                    contaminated_hashes.update(_text_hash(t) for t in df_tr["text"].fillna(""))
            except Exception:
                pass

    # 2. Mark all pool_v4.parquet rows as contaminated (seen by Part-2 LoRAs, DeBERTa, instyle)
    for pat in ["pool_v4.parquet", "pool_v1.parquet"]:
        for p in input_dir.rglob(pat):
            try:
                df_p4 = pd.read_parquet(p, columns=["text"])
                contaminated_hashes.update(_text_hash(t) for t in df_p4["text"].fillna(""))
            except Exception:
                pass

    # 3. Collect strictly unseen candidate passages from newer/disjoint corpora
    candidate_dfs = []
    seen_paths = set()
    for pat in [
        "nlm_matched_clean.parquet",
        "nlm_pool_v5.parquet",
        "nlm_pool_v2.parquet",
        "*openrouter*.parquet",
        "*openrouter*.csv",
    ]:
        for p in input_dir.rglob(pat):
            if p in seen_paths:
                continue
            seen_paths.add(p)
            try:
                if p.suffix == ".parquet":
                    df_c = pd.read_parquet(p)
                elif p.suffix == ".csv":
                    df_c = pd.read_csv(p)
                else:
                    continue

                if "text" in df_c.columns and "model" in df_c.columns:
                    keep_cols = [c for c in ["id", "prompt_id", "text", "model"] if c in df_c.columns]
                    df_c = df_c[keep_cols].dropna(subset=["text", "model"]).copy()
                    df_c = df_c[df_c["model"].isin(LABELS)]
                    df_c["_hash"] = [_text_hash(t) for t in df_c["text"]]
                    df_c = df_c[~df_c["_hash"].isin(contaminated_hashes)]
                    if len(df_c) > 0:
                        candidate_dfs.append(df_c)
            except Exception:
                pass

    if not candidate_dfs:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model", "_hash"]), set()

    df_pool_unseen = pd.concat(candidate_dfs, ignore_index=True).drop_duplicates(subset=["_hash"])

    # Stratified balanced sampling across all 18 classes
    rng = np.random.RandomState(seed)
    sampled_parts = []
    for m in LABELS:
        sub = df_pool_unseen[df_pool_unseen["model"] == m]
        if len(sub) > 0:
            n_take = min(samples_per_class, len(sub))
            idx = rng.choice(sub.index.values, size=n_take, replace=False)
            sampled_parts.append(sub.loc[idx])

    if not sampled_parts:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model", "_hash"]), set()

    df_unseen = pd.concat(sampled_parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    if "id" not in df_unseen.columns:
        df_unseen["id"] = [f"unseen_{i}" for i in range(len(df_unseen))]

    # Compute valid stopword TF-IDF average-linkage clusters (t=0.85) so prompt_id is positive integer 1..G
    if len(df_unseen) >= 2:
        try:
            tfidf = TfidfVectorizer(stop_words="english", max_features=10000, sublinear_tf=True)
            X_tfidf = tfidf.fit_transform(df_unseen["text"].fillna("").tolist())
            D = 1.0 - (X_tfidf @ X_tfidf.T).toarray().clip(0.0, 1.0)
            np.fill_diagonal(D, 0.0)
            cond_d = squareform(D, checks=False)
            clusters = fcluster(linkage(cond_d, method="average"), t=0.85, criterion="distance")
        except Exception:
            clusters = np.arange(1, len(df_unseen) + 1, dtype=int)
    else:
        clusters = np.ones(len(df_unseen), dtype=int)

    df_unseen["cluster_id"] = clusters
    df_unseen["prompt_id"] = clusters

    holdout_hashes = set(df_unseen["_hash"].tolist())
    print(
        f"[Unseen Benchmark] Built strictly unseen holdout set: {len(df_unseen)} rows "
        f"across {df_unseen['model'].nunique()} classes (0% overlap with train.csv & pool_v4.parquet)."
    )
    return df_unseen, holdout_hashes

class UnseenStepwiseAuditor:
    """
    Evaluates every model on the strictly unseen holdout dataset (`df_unseen`),
    reporting:
    1. Table 1 — Standalone (Solo) Unseen Balanced Accuracy (sorted descending).
    2. Table 2 — Greedy Forward Stepwise Selection (optimal combinations, stopping when lift < 0.0).
    3. Table 3 — Exhaustive All-2^M Subset Search & Leave-One-Out (LOO) Attribution.
    """
    def __init__(self, y_unseen: np.ndarray, groups_unseen: np.ndarray, N_test: int):
        self.y_unseen = np.asarray(y_unseen, dtype=int)
        self.groups_unseen = np.asarray(groups_unseen, dtype=int)
        self.N_test = int(N_test)
        self.models: List[Dict[str, Any]] = []

    def add_model(
        self,
        name: str,
        tier: str,
        weight: float,
        unseen_log_probs: np.ndarray,
        test_log_probs: np.ndarray,
    ):
        if len(self.y_unseen) > 0 and len(unseen_log_probs) != len(self.y_unseen):
            return
        u_lp = log_softmax(np.asarray(unseen_log_probs, dtype=np.float64), axis=1) if len(self.y_unseen) > 0 and len(unseen_log_probs) == len(self.y_unseen) else np.zeros((0, K), dtype=np.float64)
        t_lp = log_softmax(np.asarray(test_log_probs, dtype=np.float64), axis=1) if len(test_log_probs) == self.N_test else np.zeros((self.N_test, K), dtype=np.float64)
        solo_raw = float(balanced_accuracy_score(self.y_unseen, u_lp.argmax(axis=1))) if len(self.y_unseen) > 0 else 0.0
        solo_sink = _fast_sinkhorn_bacc(u_lp, self.y_unseen, iters=200) if len(self.y_unseen) >= 18 else solo_raw
        self.models.append({
            "name": name,
            "tier": tier,
            "weight": float(weight),
            "unseen_lp": u_lp,
            "test_lp": t_lp,
            "solo_unseen_bacc": solo_raw,
            "solo_unseen_sinkhorn_bacc": solo_sink,
        })

    def optimize_weights_and_blend(
        self,
        working_dir: Path,
        extra_test_log_probs: Optional[List[Tuple[str, float, np.ndarray]]] = None,
        optimize_on_unseen: bool = True,
    ) -> Tuple[np.ndarray, Dict[str, float], pd.DataFrame]:
        """
        1. Runs Greedy Forward Stepwise Selection + Coordinate Line Search on `df_unseen`
           to determine each model's optimal individual weight `w_star` (>= 0.0).
        2. Any model that fails to improve unseen balanced accuracy gets `w_star = 0.0` (PRUNED).
        3. Combines all active models + extra_test_log_probs (wpca_dp, spectral_bank) directly
           in flat log-probability space:
               LOGIT_FLAT = sum_m (w_star[m] * test_lp[m])
        """
        if len(self.models) == 0:
            return np.ones((self.N_test, K), dtype=np.float64) / float(K), {}, pd.DataFrame()

        if not optimize_on_unseen or len(self.y_unseen) < 36:
            weights_map = {m["name"]: m["weight"] for m in self.models}
            logit_flat = np.zeros((self.N_test, K), dtype=np.float64)
            for m in self.models:
                logit_flat += m["weight"] * m["test_lp"]
            if extra_test_log_probs:
                for ex_name, w_ex, lp_ex in extra_test_log_probs:
                    if len(lp_ex) == self.N_test and w_ex > 0.0:
                        logit_flat += float(w_ex) * log_softmax(lp_ex, axis=1)
                        weights_map[ex_name] = float(w_ex)
            P_mix = softmax(logit_flat, axis=1)
            df_rep = self.print_and_save_report(working_dir, final_test_preds=P_mix.argmax(axis=1))
            return P_mix, weights_map, df_rep

        M = len(self.models)
        order = sorted(range(M), key=lambda i: self.models[i]["solo_unseen_sinkhorn_bacc"], reverse=True)

        active_weights = np.zeros(M, dtype=np.float64)
        first_idx = order[0]
        active_weights[first_idx] = self.models[first_idx]["weight"]
        cum_u_lp = active_weights[first_idx] * self.models[first_idx]["unseen_lp"]
        best_bacc = float(
            balanced_accuracy_score(
                self.y_unseen,
                sinkhorn_balancing(softmax(cum_u_lp, axis=1), iters=60).argmax(axis=1),
            )
        )

        multipliers = [0.25, 0.50, 0.75, 1.00, 1.25, 1.50]
        for idx in order[1:]:
            prior_w = self.models[idx]["weight"]
            u_lp = self.models[idx]["unseen_lp"]
            best_m_w = 0.0
            best_m_bacc = best_bacc
            for mult in multipliers:
                cand_w = prior_w * mult
                trial_lp = cum_u_lp + cand_w * u_lp
                trial_P = sinkhorn_balancing(softmax(trial_lp, axis=1), iters=60)
                score = float(balanced_accuracy_score(self.y_unseen, trial_P.argmax(axis=1)))
                if score > best_m_bacc + 1e-6:
                    best_m_bacc = score
                    best_m_w = cand_w
            if best_m_w > 0.0:
                active_weights[idx] = best_m_w
                cum_u_lp = cum_u_lp + best_m_w * u_lp
                best_bacc = best_m_bacc
            else:
                active_weights[idx] = 0.0

        weights_map: Dict[str, float] = {}
        logit_flat = np.zeros((self.N_test, K), dtype=np.float64)
        for idx, m in enumerate(self.models):
            m["prior_weight"] = m["weight"]
            m["weight"] = float(active_weights[idx])
            weights_map[m["name"]] = float(active_weights[idx])
            if active_weights[idx] > 0.0:
                logit_flat += active_weights[idx] * m["test_lp"]

        if extra_test_log_probs:
            for ex_name, w_ex, lp_ex in extra_test_log_probs:
                if len(lp_ex) == self.N_test and w_ex > 0.0:
                    logit_flat += float(w_ex) * log_softmax(lp_ex, axis=1)
                    weights_map[ex_name] = float(w_ex)

        P_mix = softmax(logit_flat, axis=1)
        df_rep = self.print_and_save_report(working_dir, final_test_preds=P_mix.argmax(axis=1))
        return P_mix, weights_map, df_rep

    def evaluate_marginal_lift(
        self,
        candidate_unseen_lp: np.ndarray,
        weight: float = 1.0,
    ) -> float:
        """
        Calculates the marginal change in balanced accuracy on `df_unseen`
        if `candidate_unseen_lp` with `weight` were added to the currently registered models.
        Returns Delta BAcc.
        """
        if len(self.y_unseen) == 0 or len(candidate_unseen_lp) != len(self.y_unseen) or len(self.models) == 0:
            return 0.0

        cum_unseen_lp = np.zeros((len(self.y_unseen), K), dtype=np.float64)
        for m in self.models:
            cum_unseen_lp += m["weight"] * m["unseen_lp"]

        cur_bacc = _fast_sinkhorn_bacc(cum_unseen_lp, self.y_unseen, iters=40)

        cand_lp = log_softmax(np.asarray(candidate_unseen_lp, dtype=np.float64), axis=1)
        new_unseen_lp = cum_unseen_lp + float(weight) * cand_lp
        new_bacc = _fast_sinkhorn_bacc(new_unseen_lp, self.y_unseen, iters=40)

        return new_bacc - cur_bacc

    def print_and_save_report(self, working_dir: Path, final_test_preds: Optional[np.ndarray] = None) -> pd.DataFrame:
        if not self.models or len(self.y_unseen) == 0:
            print("[Ablation] No models or unseen holdout samples available; skipping report.")
            return pd.DataFrame()

        # Part 1: Standalone (Solo) Accuracy, sorted descending
        print("\n" + "=" * 105)
        print(f" PART 1: STANDALONE (SOLO) ACCURACY ON STRICTLY UNSEEN HOLDOUT ({len(self.y_unseen)} Disjoint Passages)")
        print("=" * 105)
        print(f"{'Model / Checkpoint':<28} | {'Tier':<6} | {'Unseen Raw BAcc':<16} | {'Unseen+Sinkhorn BAcc':<20} | {'Agrees w/ Final Test'}")
        print("-" * 105)
        sorted_solo = sorted(self.models, key=lambda m: m["solo_unseen_sinkhorn_bacc"], reverse=True)
        for m in sorted_solo:
            agree_str = "N/A"
            if final_test_preds is not None and len(final_test_preds) == self.N_test:
                pct = 100.0 * float((m["test_lp"].argmax(axis=1) == final_test_preds).mean())
                agree_str = f"{pct:5.1f}%"
            print(
                f"{m['name']:<28} | {m['tier']:<6} | "
                f"{m['solo_unseen_bacc']:<16.4f} | {m['solo_unseen_sinkhorn_bacc']:<20.4f} | {agree_str}"
            )

        # Part 2: Greedy Forward Stepwise Selection (starts with strongest solo model, tests all candidates at each step)
        print("\n" + "=" * 105)
        print(" PART 2: GREEDY FORWARD STEPWISE SELECTION ON STRICTLY UNSEEN HOLDOUT (Optimal Combinations)")
        print("=" * 105)
        print(f"{'Step':<5} | {'Added Model':<28} | {'Wt':<5} | {'Cumul Raw':<10} | {'Cumul+Sink':<11} | {'Marginal Lift':<13} | {'Test Rows Flipped'}")
        print("-" * 105)

        M = len(self.models)
        remaining = list(range(M))
        cum_unseen_lp = np.zeros((len(self.y_unseen), K), dtype=np.float64)
        cum_test_lp = np.zeros((self.N_test, K), dtype=np.float64)
        prev_bacc = 0.0
        prev_test_preds = None
        records = []

        step_idx = 1
        while remaining:
            best_cand_idx = None
            best_cand_bacc = -1.0
            best_cand_raw = 0.0

            if step_idx == 1:
                # Pick model with highest solo unseen sinkhorn bacc
                for idx in remaining:
                    cand = self.models[idx]
                    cand_lp = cand["weight"] * cand["unseen_lp"]
                    cand_bacc = _fast_sinkhorn_bacc(cand_lp, self.y_unseen, iters=40)
                    if cand_bacc > best_cand_bacc:
                        best_cand_bacc = cand_bacc
                        best_cand_idx = idx
                        best_cand_raw = float(balanced_accuracy_score(self.y_unseen, cand_lp.argmax(axis=1)))
            else:
                # Test all remaining candidate models
                for idx in remaining:
                    cand = self.models[idx]
                    cand_lp = cum_unseen_lp + cand["weight"] * cand["unseen_lp"]
                    cand_bacc = _fast_sinkhorn_bacc(cand_lp, self.y_unseen, iters=40)
                    if cand_bacc > best_cand_bacc:
                        best_cand_bacc = cand_bacc
                        best_cand_idx = idx
                        best_cand_raw = float(balanced_accuracy_score(self.y_unseen, cand_lp.argmax(axis=1)))

            if best_cand_idx is None:
                break

            lift = best_cand_bacc - prev_bacc if step_idx > 1 else 0.0
            if step_idx > 1 and lift < 0.0:
                print(f"[Greedy Forward Selection] Best remaining candidate '{self.models[best_cand_idx]['name']}' gives negative lift ({lift:+.4f}); stopping forward selection.")
                break

            m = self.models[best_cand_idx]
            remaining.remove(best_cand_idx)

            w = m["weight"]
            cum_unseen_lp = cum_unseen_lp + w * m["unseen_lp"]
            cum_test_lp = cum_test_lp + w * m["test_lp"]

            lift_str = f"{lift:+.4f}" if step_idx > 1 else "BASELINE"
            cur_test_preds = sinkhorn_balancing(softmax(cum_test_lp, axis=1), iters=40).argmax(axis=1) if self.N_test > 0 else np.array([])
            if prev_test_preds is None:
                flips = self.N_test
                flip_str = f"{flips} (Init)"
            else:
                flips = int((cur_test_preds != prev_test_preds).sum()) if len(cur_test_preds) == len(prev_test_preds) else 0
                flip_pct = 100.0 * flips / max(1, self.N_test)
                flip_str = f"{flips:4d} ({flip_pct:4.1f}%)"

            print(
                f"{step_idx:<5} | + {m['name']:<26} | {w:<5.2f} | "
                f"{best_cand_raw:<10.4f} | {best_cand_bacc:<11.4f} | {lift_str:<13} | {flip_str}"
            )
            prev_bacc = best_cand_bacc
            prev_test_preds = cur_test_preds
            records.append({
                "step": step_idx,
                "model": m["name"],
                "added_model": m["name"],
                "tier": m["tier"],
                "weight": w,
                "solo_unseen_bacc": m["solo_unseen_bacc"],
                "solo_unseen_sinkhorn_bacc": m["solo_unseen_sinkhorn_bacc"],
                "cumul_unseen_raw_bacc": best_cand_raw,
                "cumul_unseen_sinkhorn_bacc": best_cand_bacc,
                "marginal_lift": lift,
                "test_rows_flipped": flips,
            })
            step_idx += 1

        print("=" * 105 + "\n")

        # Part 3: Exhaustive All 2^M Subset Search & LOO Attribution (for M <= 12)
        if 2 <= M <= 12:
            print("=" * 105)
            print(f" PART 3: GLOBALLY OPTIMAL SUBSET & LEAVE-ONE-OUT (LOO) ATTRIBUTION (All {2**M - 1} Non-Empty Subsets)")
            print("=" * 105)

            best_subset = None
            best_subset_bacc = -1.0

            for r in range(1, M + 1):
                for sub in combinations(range(M), r):
                    sub_unseen_lp = sum(self.models[i]["weight"] * self.models[i]["unseen_lp"] for i in sub)
                    bacc = _fast_sinkhorn_bacc(sub_unseen_lp, self.y_unseen, iters=30)
                    if bacc > best_subset_bacc:
                        best_subset_bacc = bacc
                        best_subset = sub

            opt_names = [self.models[i]["name"] for i in best_subset]
            print(f"Globally Optimal Ensemble BAcc: {best_subset_bacc:.4f} (Uses {len(best_subset)} / {M} models)")
            print(f"Optimal Set: {', '.join(opt_names)}")
            print("-" * 105)
            print(f"{'Model / Checkpoint':<28} | {'In Optimal?':<12} | {'Weight':<7} | {'Delta_LOO (BAcc Drop If Excluded)'}")
            print("-" * 105)

            opt_set = set(best_subset)
            for i, m in enumerate(self.models):
                in_opt = i in opt_set
                in_str = "YES" if in_opt else "NO (Pruned)"
                if in_opt:
                    if len(best_subset) == 1:
                        delta_loo = best_subset_bacc
                    else:
                        loo_sub = [j for j in best_subset if j != i]
                        loo_lp = sum(self.models[j]["weight"] * self.models[j]["unseen_lp"] for j in loo_sub)
                        loo_bacc = _fast_sinkhorn_bacc(loo_lp, self.y_unseen, iters=30)
                        delta_loo = best_subset_bacc - loo_bacc
                    loo_str = f"-{delta_loo:.4f}" if delta_loo >= 0 else f"+{abs(delta_loo):.4f}"
                else:
                    with_m_sub = list(best_subset) + [i]
                    with_m_lp = sum(self.models[j]["weight"] * self.models[j]["unseen_lp"] for j in with_m_sub)
                    with_m_bacc = _fast_sinkhorn_bacc(with_m_lp, self.y_unseen, iters=30)
                    delta_add = with_m_bacc - best_subset_bacc
                    loo_str = f"{delta_add:+.4f} (Degrades if added)"

                print(f"{m['name']:<28} | {in_str:<12} | {m['weight']:<7.2f} | {loo_str}")

                for r in records:
                    if r["model"] == m["name"]:
                        r["in_optimal_subset"] = in_opt
                        r["delta_loo"] = delta_loo if in_opt else 0.0

            print("=" * 105 + "\n")

        df_rep = pd.DataFrame(records)
        working_dir = Path(working_dir)
        working_dir.mkdir(parents=True, exist_ok=True)
        df_rep.to_csv(working_dir / "unseen_stepwise_ablation.csv", index=False)
        return df_rep


In [ ]:
# === MODULE: src/namethatllm/trainer.py ===
"""
Two-Stage Multi-Layer LoRA Training Engine
==========================================
Implements production Two-Stage Multi-Layer LoRA training with 8 intermediate
LayerNorm heads, causal attention-sink masking (m[:, 0] = 0.0), prompt-grouped
contrastive centering, and standard checkpoint export.
"""

import os
import gc
import json
import math
import time
import importlib.util
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any, Iterator

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Neutralize Kaggle's incompatible pre-installed torchao (<0.16.0) across all libraries
_orig_find_spec = importlib.util.find_spec
def _safe_find_spec(name, *args, **kwargs):
    if name == "torchao":
        return None
    return _orig_find_spec(name, *args, **kwargs)
importlib.util.find_spec = _safe_find_spec

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler


# Disable incompatible torchao version check in Kaggle environments
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass

def _disable_final_norm(mod: nn.Module) -> None:
    for path in ["norm", "model.norm", "base_model.norm", "base_model.model.norm", "base_model.model.model.norm"]:
        curr = mod
        parts = path.split(".")
        for p in parts[:-1]:
            curr = getattr(curr, p, None)
            if curr is None:
                break
        if curr is not None and hasattr(curr, parts[-1]) and getattr(curr, parts[-1]) is not None:
            setattr(curr, parts[-1], nn.Identity())

def _find_decoder_layer_list(backbone: nn.Module) -> Optional[nn.ModuleList]:
    """Finds the main sequence of transformer/decoder layer modules within a model."""
    for path in [
        "model.layers",
        "base_model.model.layers",
        "base_model.model.model.layers",
        "layers",
        "encoder.layer",
        "deberta.encoder.layer",
        "base_model.encoder.layer",
        "transformer.h",
        "base_model.transformer.h",
    ]:
        curr = backbone
        ok = True
        for p in path.split("."):
            if hasattr(curr, p):
                curr = getattr(curr, p)
            else:
                ok = False
                break
        if ok and isinstance(curr, (nn.ModuleList, list, tuple)):
            return curr
    return None

class MultiLayerLoRAClassifier(nn.Module):
    """
    Universal Multi-Layer LoRA sequence classifier with dedicated per-layer
    LayerNorm heads, causal attention-sink masking (m[:, 0] = 0.0), and
    5-branch Multi-Sample Dropout.
    """
    def __init__(
        self,
        backbone: nn.Module,
        hidden_size: int,
        layers: List[int],
        num_classes: int = K,
        keep_layers: Optional[int] = None
    ):
        super().__init__()
        self.backbone = backbone
        _disable_final_norm(self.backbone)
        self.layers = list(layers)
        self.num_classes = num_classes
        self.hidden_size = hidden_size
        self.total_dim = len(layers) * hidden_size

        # Dedicated LayerNorm for each intermediate layer
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_size) for _ in layers])

        # Dual classification heads (Plain + Prompt-Centered Twin)
        self.plain_head = nn.Linear(self.total_dim, num_classes)
        self.centered_head = nn.Linear(self.total_dim, num_classes)

        # 5-branch Multi-Sample Dropout for superior regularization
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in (0.1, 0.2, 0.3, 0.4, 0.5)])

    def predict_plain(self, Z: torch.Tensor) -> torch.Tensor:
        first_dev = Z.device
        if next(self.plain_head.parameters()).device != first_dev:
            self.plain_head = self.plain_head.to(first_dev)
        if self.training:
            return torch.mean(torch.stack([self.plain_head(d(Z)) for d in self.dropouts], dim=0), dim=0)
        return self.plain_head(Z)

    def predict_centered(self, Z: torch.Tensor) -> torch.Tensor:
        first_dev = Z.device
        if next(self.centered_head.parameters()).device != first_dev:
            self.centered_head = self.centered_head.to(first_dev)
        if self.training:
            return torch.mean(torch.stack([self.centered_head(d(Z)) for d in self.dropouts], dim=0), dim=0)
        return self.centered_head(Z)

    def forward_features(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        drop_first_token: bool = True
    ) -> torch.Tensor:
        """Forward pass with causal attention sink masking and multi-layer LayerNorm pooling."""
        m = attention_mask.clone().float()
        if drop_first_token and m.shape[1] > 1:
            m[:, 0] = 0.0
        den = m.sum(dim=1, keepdim=True).clamp(min=1e-5)
        first_device = input_ids.device

        dec_layers = _find_decoder_layer_list(self.backbone)
        can_use_hooks = (
            dec_layers is not None
            and all((0 <= (l - 1) < len(dec_layers)) for l in self.layers)
        )

        if can_use_hooks:
            hooked_pooled = {}
            handles = []
            for idx, l in enumerate(self.layers):
                norm = self.norms[idx]
                layer_mod = dec_layers[l - 1]

                def _make_hook(norm_layer, l_idx):
                    def _hook(mod, inp, out):
                        h = out[0] if isinstance(out, tuple) else out
                        h = h.float()
                        ldev = h.device
                        m_l = m.to(ldev)
                        den_l = den.to(ldev)
                        h = torch.nan_to_num(h, nan=0.0, posinf=1e4, neginf=-1e4)
                        pooled = (h * m_l.unsqueeze(-1)).sum(dim=1) / den_l
                        if next(norm_layer.parameters()).device != ldev:
                            norm_layer.to(ldev)
                        normed = norm_layer(pooled)
                        hooked_pooled[l_idx] = normed.to(first_device)
                    return _hook

                handles.append(layer_mod.register_forward_hook(_make_hook(norm, idx)))

            try:
                try:
                    self.backbone(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        output_hidden_states=False,
                        return_dict=True,
                        use_cache=False
                    )
                except TypeError:
                    self.backbone(input_ids=input_ids, attention_mask=attention_mask)
            finally:
                for h in handles:
                    h.remove()

            if len(hooked_pooled) == len(self.layers):
                pooled_list = [hooked_pooled[idx] for idx in range(len(self.layers))]
                return torch.cat(pooled_list, dim=-1)

        # Fallback to output_hidden_states=True
        try:
            outputs = self.backbone(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                return_dict=True,
                use_cache=False
            )
        except TypeError:
            outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        hidden_states = getattr(outputs, "hidden_states", outputs)
        pooled_list = []
        for idx, l in enumerate(self.layers):
            h = hidden_states[l].float()
            layer_dev = h.device
            m_l = m.to(layer_dev)
            den_l = den.to(layer_dev)
            h = torch.nan_to_num(h, nan=0.0, posinf=1e4, neginf=-1e4)
            pooled = (h * m_l.unsqueeze(-1)).sum(dim=1) / den_l
            norm = self.norms[idx]
            if next(norm.parameters()).device != layer_dev:
                norm = norm.to(layer_dev)
                self.norms[idx] = norm
            normed = norm(pooled)
            pooled_list.append(normed.to(first_device))

        return torch.cat(pooled_list, dim=-1)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        drop_first_token: bool = True
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        Z = self.forward_features(input_ids, attention_mask, drop_first_token=drop_first_token)
        logits_plain = self.predict_plain(Z)
        logits_centered = self.predict_centered(Z)
        return Z, logits_plain, logits_centered

def _build_sister_mask(device: torch.device) -> torch.Tensor:
    """Builds an (18, 18) binary mask of sister models within the same model family."""
    M = torch.zeros((K, K), dtype=torch.float32, device=device)
    for fam_name, members in FAMILIES.items():
        for m1 in members:
            for m2 in members:
                if m1 != m2:
                    M[m1, m2] = 1.0
    return M

def family_margin_cross_entropy(
    logits: torch.Tensor,
    labels: torch.Tensor,
    family_margin: float = 0.35,
    label_smoothing: float = 0.0
) -> torch.Tensor:
    """
    Computes cross-entropy with intra-family margin penalty.
    Sister models within FAMILIES receive an additive logit penalty (+delta),
    forcing the network to establish a decisive decision boundary against sister LLMs.
    """
    if family_margin > 0.0:
        mask = _build_sister_mask(logits.device)
        boost = mask[labels] * family_margin
        logits = logits + boost
    return F.cross_entropy(logits, labels, label_smoothing=label_smoothing)

def compute_supcon_loss(
    Z_c_mb: torch.Tensor,
    labels_mb: torch.Tensor,
    centered_head_weight: torch.Tensor,
    Z_group_centered: Optional[torch.Tensor] = None,
    temperature: float = 0.10,
) -> torch.Tensor:
    """
    Prototype-anchored supervised contrastive loss on prompt-centered embeddings Z_c_mb:
    - Positive anchor for sample i is its L2-normalized class prototype w_{y_i} from centered_head.
    - Negatives are the other 17 class prototypes + all other passages in the current prompt group.
    Guarantees well-defined, non-NaN gradients even when micro_batch_size=2 and all prompt labels are unique!
    """
    z_norm = F.normalize(Z_c_mb.float(), p=2, dim=-1, eps=1e-6)
    w_norm = F.normalize(centered_head_weight.float().to(z_norm.device), p=2, dim=-1, eps=1e-6)
    sim_proto = (z_norm @ w_norm.T) / max(float(temperature), 1e-3)
    loss = F.cross_entropy(sim_proto, labels_mb.to(z_norm.device))

    if Z_group_centered is not None and Z_group_centered.shape[0] > Z_c_mb.shape[0]:
        g_norm = F.normalize(Z_group_centered.detach().float().to(z_norm.device), p=2, dim=-1, eps=1e-6)
        # Repel distinct passages within the same prompt group (orthogonality / uniformity regularization)
        sim_group = z_norm @ g_norm.T
        loss = loss + 0.10 * (sim_group.pow(2).mean())
    return loss

def compute_two_stage_loss(
    model: MultiLayerLoRAClassifier,
    Z: torch.Tensor,
    labels: torch.Tensor,
    prompt_ids: Optional[List[Any]] = None,
    centered_weight: float = 0.5,
    family_margin: float = 0.35,
    supcon_weight: float = 0.0,
    label_smoothing: float = 0.0
) -> Tuple[torch.Tensor, torch.Tensor, Optional[torch.Tensor]]:
    """
    Computes joint plain cross-entropy and within-prompt contrastive centered loss
    with intra-family margin penalties and optional supervised contrastive loss:
    L = L_plain + centered_weight * L_centered + supcon_weight * L_supcon
    """
    dev = Z.device
    labels_dev = labels.to(dev)

    if hasattr(model, "predict_plain"):
        logits_plain = model.predict_plain(Z)
    else:
        if next(model.plain_head.parameters()).device != dev:
            model.plain_head = model.plain_head.to(dev)
        logits_plain = model.plain_head(Z)

    loss_plain = family_margin_cross_entropy(logits_plain, labels_dev, family_margin=family_margin, label_smoothing=label_smoothing)

    if prompt_ids is None or centered_weight <= 0.0:
        return loss_plain, loss_plain, None

    # Exact within-prompt centering
    Z_c = Z.clone()
    prompt_arr = np.array(prompt_ids)
    unique_prompts = list(dict.fromkeys(prompt_ids))
    for p in unique_prompts:
        mask = torch.tensor(prompt_arr == p, device=dev, dtype=torch.bool)
        if mask.sum() > 1:
            Z_c[mask] = Z[mask] - Z[mask].mean(dim=0, keepdim=True)
        else:
            Z_c[mask] = Z[mask] - Z[mask]

    if hasattr(model, "predict_centered"):
        logits_centered = model.predict_centered(Z_c)
    else:
        if next(model.centered_head.parameters()).device != dev:
            model.centered_head = model.centered_head.to(dev)
        logits_centered = model.centered_head(Z_c)

    loss_centered = family_margin_cross_entropy(logits_centered, labels_dev, family_margin=family_margin, label_smoothing=label_smoothing)

    total_loss = loss_plain + float(centered_weight) * loss_centered
    if supcon_weight > 0.0:
        loss_sup = compute_supcon_loss(Z_c, labels_dev, model.centered_head.weight, Z_group_centered=None)
        total_loss = total_loss + float(supcon_weight) * loss_sup

    return total_loss, loss_plain, loss_centered

class PromptGroupBatchSampler(Sampler):
    """
    Groups samples sharing the same prompt_id into complete batches (one prompt group per step)
    to enable two-pass micro-batched within-prompt contrastive centering during Stage 2 training.
    """
    def __init__(self, prompt_ids: List[Any], batch_size: int = 18, shuffle: bool = True, seed: int = 42):
        self.prompt_to_indices: Dict[Any, List[int]] = {}
        for idx, p in enumerate(prompt_ids):
            self.prompt_to_indices.setdefault(p, []).append(idx)
        self.prompts = list(self.prompt_to_indices.keys())
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def __iter__(self) -> Iterator[List[int]]:
        rng = np.random.RandomState(self.seed + self.epoch)
        prompts = list(self.prompts)
        if self.shuffle:
            rng.shuffle(prompts)
        for p in prompts:
            idxs = list(self.prompt_to_indices[p])
            if self.shuffle:
                rng.shuffle(idxs)
            yield idxs

    def __len__(self) -> int:
        return len(self.prompts)

    def set_epoch(self, epoch: int):
        self.epoch = epoch

class TextSequenceDataset(Dataset):
    def __init__(
        self,
        texts: List[str],
        labels: Optional[List[int]],
        prompt_ids: Optional[List[Any]],
        tok: Any,
        max_len: int = 447,
        end_id: int = 151643
    ):
        self.texts = texts
        self.labels = labels
        self.prompt_ids = prompt_ids
        self.tok = tok
        self.max_len = max_len
        self.end_id = end_id

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        raw_ids = self.tok(str(self.texts[idx]), add_special_tokens=False, truncation=True, max_length=self.max_len)["input_ids"]
        input_ids = raw_ids + [self.end_id]
        item: Dict[str, Any] = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
        }
        if self.labels is not None:
            item["label"] = torch.tensor(self.labels[idx], dtype=torch.long)
        if self.prompt_ids is not None:
            item["prompt_id"] = self.prompt_ids[idx]
        return item

def collate_text_batch(batch: List[Dict[str, Any]], pad_token_id: int = 151643) -> Dict[str, Any]:
    input_ids = [item["input_ids"] for item in batch]
    masks = [item.get("attention_mask", torch.ones_like(item["input_ids"])) for item in batch]
    padded_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
    padded_mask = torch.nn.utils.rnn.pad_sequence(masks, batch_first=True, padding_value=0)
    res: Dict[str, Any] = {"input_ids": padded_ids, "attention_mask": padded_mask}
    if "label" in batch[0]:
        res["labels"] = torch.stack([item["label"] for item in batch])
    if "prompt_id" in batch[0]:
        res["prompt_ids"] = [item["prompt_id"] for item in batch]
    return res

class BucketedLengthBatchSampler(Sampler):
    """
    Batches sequences of similar lengths together to minimize padding waste,
    while shuffling batches for stochastic gradient descent.
    """
    def __init__(self, lengths: List[int], batch_size: int, bucket_size_multiplier: int = 50, shuffle: bool = True, seed: int = 42):
        self.lengths = lengths
        self.batch_size = batch_size
        self.bucket_size = batch_size * bucket_size_multiplier
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def __iter__(self) -> Iterator[List[int]]:
        rng = np.random.RandomState(self.seed + self.epoch)
        indices = np.arange(len(self.lengths))
        if self.shuffle:
            rng.shuffle(indices)

        batches = []
        for i in range(0, len(indices), self.bucket_size):
            bucket = indices[i:i + self.bucket_size]
            sorted_bucket = bucket[np.argsort([self.lengths[idx] for idx in bucket])]
            for b in range(0, len(sorted_bucket), self.batch_size):
                batches.append(list(sorted_bucket[b:b + self.batch_size]))

        if self.shuffle:
            rng.shuffle(batches)
        for batch in batches:
            yield batch

    def __len__(self) -> int:
        return math.ceil(len(self.lengths) / self.batch_size)

    def set_epoch(self, epoch: int):
        self.epoch = epoch

def train_prompt_group_step(
    model: MultiLayerLoRAClassifier,
    opt: torch.optim.Optimizer,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    labels: torch.Tensor,
    micro_batch_size: int = 2,
    centered_weight: float = 0.5,
    family_margin: float = 0.35,
    supcon_weight: float = 0.20,
    label_smoothing: float = 0.10
) -> Tuple[float, float, float]:
    """
    Two-pass micro-batched training step for a single prompt group.
    Pass 1: Computes exact within-prompt mean vector mu_prompt without gradients in chunks of 4.
    Pass 2: Computes plain and centered cross-entropy losses over micro-batches of micro_batch_size,
            accumulating gradients directly with dynamic sequence length slicing, AMP fp16,
            family margin penalties, and supervised contrastive loss.
    """
    N = input_ids.shape[0]
    dev = input_ids.device
    labels_dev = labels.to(dev)

    # Pass 1: Compute mu_prompt with no_grad in chunks of 4
    z_chunks = []
    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=torch.float16):
            for s in range(0, N, 4):
                e = min(s + 4, N)
                mb_mask = attention_mask[s:e]
                max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
                z_c = model.forward_features(input_ids[s:e, :max_l], mb_mask[:, :max_l], drop_first_token=True)
                z_chunks.append(z_c)
    Z_all_no_grad = torch.cat(z_chunks, dim=0)
    mu_prompt = Z_all_no_grad.mean(dim=0, keepdim=True)
    Z_group_centered = (Z_all_no_grad - mu_prompt).detach()
    del z_chunks

    # Pass 2: Autograd micro-batches with AMP
    total_loss_val = 0.0
    plain_loss_val = 0.0
    centered_loss_val = 0.0

    model.train()
    for s in range(0, N, micro_batch_size):
        e = min(s + micro_batch_size, N)
        mb_mask = attention_mask[s:e]
        max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
        mb_ids = input_ids[s:e, :max_l]
        mb_mask_slice = mb_mask[:, :max_l]

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=torch.float16):
            Z_mb = model.forward_features(mb_ids, mb_mask_slice, drop_first_token=True)
            mb_dev = Z_mb.device

            logits_p = model.predict_plain(Z_mb)
            loss_p = family_margin_cross_entropy(logits_p, labels_dev[s:e].to(mb_dev), family_margin=family_margin, label_smoothing=label_smoothing)

            Z_c_mb = Z_mb - mu_prompt.to(mb_dev)
            logits_c = model.predict_centered(Z_c_mb)
            loss_c = family_margin_cross_entropy(logits_c, labels_dev[s:e].to(mb_dev), family_margin=family_margin, label_smoothing=label_smoothing)

            loss_mb = loss_p + float(centered_weight) * loss_c
            if supcon_weight > 0.0:
                loss_sup = compute_supcon_loss(
                    Z_c_mb,
                    labels_dev[s:e],
                    model.centered_head.weight,
                    Z_group_centered=Z_group_centered,
                    temperature=0.10
                )
                loss_mb = loss_mb + float(supcon_weight) * loss_sup

            weight_frac = (e - s) / float(N)
            scaled_loss = loss_mb * weight_frac

        scaled_loss.backward()

        total_loss_val += loss_mb.item() * weight_frac
        plain_loss_val += loss_p.item() * weight_frac
        centered_loss_val += loss_c.item() * weight_frac

    return total_loss_val, plain_loss_val, centered_loss_val

def fit_ridge_heads(
    Z: np.ndarray,
    y: np.ndarray,
    prompts: Optional[np.ndarray] = None,
    alpha: float = 10.0,
    label_smoothing: float = 0.10,
    num_classes: int = K
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Fits closed-form dual Ridge regression heads for plain and centered representations
    with label smoothing. Uses dual formulation (N x N) for ultra-fast, numerically stable solution.
    """
    N, D = Z.shape
    Y = np.zeros((N, num_classes), dtype=np.float32)
    for i, lab in enumerate(y):
        Y[i, lab] = 1.0
    if label_smoothing > 0.0:
        Y = (1.0 - label_smoothing) * Y + (label_smoothing / num_classes)

    # Plain head: Z_tilde = [Z, 1]
    Z_tilde = np.hstack([Z, np.ones((N, 1), dtype=np.float32)])
    K_mat = Z_tilde @ Z_tilde.T
    K_mat.flat[::N + 1] += alpha
    A = np.linalg.solve(K_mat, Y)
    W_tilde = (A.T @ Z_tilde).astype(np.float32)
    W_p = W_tilde[:, :-1]
    b_p = W_tilde[:, -1]

    # Centered head: Z_c
    Z_c = Z.copy()
    if prompts is not None:
        u_p = np.unique(prompts)
        for p in u_p:
            mask = (prompts == p)
            if mask.sum() > 1:
                Z_c[mask] = Z[mask] - Z[mask].mean(axis=0, keepdims=True)
            else:
                Z_c[mask] = 0.0

    Z_c_tilde = np.hstack([Z_c, np.ones((N, 1), dtype=np.float32)])
    K_mat_c = Z_c_tilde @ Z_c_tilde.T
    K_mat_c.flat[::N + 1] += alpha
    A_c = np.linalg.solve(K_mat_c, Y)
    W_c_tilde = (A_c.T @ Z_c_tilde).astype(np.float32)
    W_c = W_c_tilde[:, :-1]
    b_c = W_c_tilde[:, -1]

    return W_p, b_p, W_c, b_c

def export_compatible_checkpoint(
    model: MultiLayerLoRAClassifier,
    output_dir: Path,
    layers: List[int],
    keep: int,
    max_len: int = 447,
    end_id: int = 151643,
    backbone_name: Optional[str] = None,
    Z_train: Optional[np.ndarray] = None,
    y_train: Optional[np.ndarray] = None,
    prompts_train: Optional[np.ndarray] = None
) -> Path:
    """
    Exports standard checkpoint artifacts strictly compatible with inference loaders:
    1. adapter/ (LoRA adapter weights and config)
    2. heads.safetensors (norms.{i}.weight, norms.{i}.bias, plain.weight, plain.bias, centered.weight, centered.bias)
    3. meta.json (keep, layers, max_len, end_id, labels, backbone)
    4. wPCA_bank.npz (shared Gaussian covariance / whitening basis)
    """
    from safetensors.torch import save_file

    out_p = Path(output_dir)
    out_p.mkdir(parents=True, exist_ok=True)

    # 1. Adapter weights
    adapter_dir = out_p / "adapter"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    if hasattr(model.backbone, "save_pretrained"):
        model.backbone.save_pretrained(str(adapter_dir))
    elif hasattr(model, "save_pretrained"):
        model.save_pretrained(str(adapter_dir))

    # 2. Heads dictionary
    heads_dict = {}
    for i, norm in enumerate(model.norms):
        heads_dict[f"norms.{i}.weight"] = norm.weight.detach().cpu().contiguous()
        if hasattr(norm, "bias") and norm.bias is not None:
            heads_dict[f"norms.{i}.bias"] = norm.bias.detach().cpu().contiguous()

    heads_dict["plain.weight"] = model.plain_head.weight.detach().cpu().contiguous()
    heads_dict["plain.bias"] = model.plain_head.bias.detach().cpu().contiguous()
    heads_dict["centered.weight"] = model.centered_head.weight.detach().cpu().contiguous()
    heads_dict["centered.bias"] = model.centered_head.bias.detach().cpu().contiguous()

    save_file(heads_dict, str(out_p / "heads.safetensors"))

    # 3. Meta JSON
    meta = {
        "labels": list(LABELS),
        "backbone": str(backbone_name) if backbone_name else "",
        "cfg": {
            "keep": int(keep),
            "layers": [int(x) for x in layers],
            "max_len": int(max_len),
            "end_token_id": int(end_id),
            "backbone": str(backbone_name) if backbone_name else ""
        }
    }
    (out_p / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    # 4. wPCA Bank precomputation
    if Z_train is not None and y_train is not None and prompts_train is not None:
        d_target = min(256, Z_train.shape[1])
        res = fit_shared_gaussian_lowram(Z_train, y_train, prompts_train, d_target=d_target)
        np.savez_compressed(
            str(out_p / "wPCA_bank.npz"),
            mu=res["mx"].astype(np.float32),
            sd=res["sx"].astype(np.float32),
            W=res["W"].astype(np.float32),
            layers=np.array(layers, dtype=np.int32)
        )

    return out_p

def run_training_pipeline(train_cfg: Optional[Dict[str, Any]] = None) -> Path:
    """
    Executes complete Two-Stage Multi-Layer LoRA training pipeline:
    Stage 1: Multi-corpus pretraining on up to 90k passages with family margin loss.
    Stage 2: Prompt-grouped contrastive fine-tuning on train.csv with SupCon loss.
    """
    from transformers import AutoConfig, AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
    from peft import LoraConfig, get_peft_model

    if train_cfg is None:
        train_cfg = {}

    backbone_name = train_cfg.get("backbone", "Qwen/Qwen3-14B-Base")
    keep_layers = int(train_cfg.get("keep_layers", 27))
    layers = list(train_cfg.get("layers", [2, 4, 6, 9, 13, 17, 22, 27]))
    max_len = int(train_cfg.get("max_len", 512))
    end_id = int(train_cfg.get("end_id", 151643))
    out_dir = Path(train_cfg.get("output_dir", "export_champion_qwen3_14b"))
    family_margin = float(train_cfg.get("family_margin", 0.35))
    supcon_weight = float(train_cfg.get("supcon_weight", 0.20))
    is_deberta = "deberta" in backbone_name.lower()

    input_dir = Path(train_cfg.get("input_dir", "/kaggle/input"))
    local_base = None
    if input_dir.exists():
        if is_deberta:
            for cfg_p in input_dir.rglob("config.json"):
                try:
                    c_data = json.load(open(cfg_p))
                    if "deberta" in str(c_data.get("model_type", "")).lower():
                        local_base = cfg_p.parent
                        break
                except Exception:
                    continue
        elif "2.5" in backbone_name.lower() or "qwen25" in backbone_name.lower():
            local_base = find_base_model(input_dir, "qwen2", 5120, 48) or find_base_model(input_dir, "qwen2", 3584, 28)
        elif "14b" in backbone_name.lower():
            local_base = find_base_model(input_dir, "qwen3", 5120, 40) or find_base_model(input_dir, "qwen2", 5120, 48)
        elif "8b" in backbone_name.lower():
            local_base = find_base_model(input_dir, "qwen3", 4096, 36)
    model_source = str(local_base) if local_base else backbone_name

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    n_gpus = torch.cuda.device_count()
    print(f"[Trainer] Device: {device}, Available GPUs: {n_gpus}, Model Source: {model_source}, Keep: {keep_layers}")

    train_path, test_path, pool_file = resolve_dataset_paths(input_dir if input_dir.exists() else Path("."))
    all_pools = resolve_all_pool_files(input_dir if input_dir.exists() else Path("."))

    tok = AutoTokenizer.from_pretrained(model_source, trust_remote_code=True, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    vocab_sz = len(tok)
    if end_id >= vocab_sz:
        safe_end_id = int(tok.eos_token_id or tok.sep_token_id or 2)
        print(f"[Trainer] Config end_id ({end_id}) >= vocab size ({vocab_sz}); adjusting to {safe_end_id}")
        end_id = safe_end_id

    base_cfg = AutoConfig.from_pretrained(model_source, trust_remote_code=True)
    base_cfg.num_hidden_layers = keep_layers

    # Multi-GPU 2xT4 memory balancing: balance across both GPUs so GPU 0 has plenty of headroom
    head_dev = torch.device("cuda:1" if n_gpus >= 2 else ("cuda:0" if n_gpus == 1 else "cpu"))
    if n_gpus >= 2:
        max_memory = {0: "13.0GiB", 1: "13.5GiB"}
        base_model = AutoModel.from_pretrained(
            model_source,
            config=base_cfg,
            torch_dtype=torch.float16,
            device_map="auto",
            max_memory=max_memory,
            trust_remote_code=True
        )
    elif n_gpus == 1:
        base_model = AutoModel.from_pretrained(
            model_source,
            config=base_cfg,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
    else:
        base_model = AutoModel.from_pretrained(
            model_source,
            config=base_cfg,
            torch_dtype=torch.float32,
            trust_remote_code=True
        )

    try:
        import peft.import_utils
        import peft.tuners.lora.torchao
        peft.import_utils.is_torchao_available = lambda: False
        peft.tuners.lora.torchao.is_torchao_available = lambda: False
    except Exception:
        pass

    _disable_final_norm(base_model)

    if hasattr(base_model, "enable_input_require_grads"):
        base_model.enable_input_require_grads()

    linear_mod_names = set()
    for name, module in base_model.named_modules():
        if isinstance(module, nn.Linear):
            linear_mod_names.add(name.split(".")[-1])

    configured_targets = list(train_cfg.get("lora_target_modules", []))
    if not configured_targets:
        configured_targets = (
            ["query_proj", "key_proj", "value_proj", "dense"]
            if is_deberta
            else ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
        )

    active_targets = [m for m in configured_targets if m in linear_mod_names]
    if not active_targets:
        active_targets = list(linear_mod_names)[:7] if linear_mod_names else configured_targets
        print(f"[Trainer] Configured LoRA targets not found; falling back to: {active_targets}")
    else:
        print(f"[Trainer] Active LoRA target modules: {active_targets}")

    lora_cfg = LoraConfig(
        r=int(train_cfg.get("lora_r", 32)),
        lora_alpha=int(train_cfg.get("lora_alpha", 64)),
        lora_dropout=float(train_cfg.get("lora_dropout", 0.05)),
        target_modules=active_targets,
        bias="none",
        task_type="FEATURE_EXTRACTION"
    )
    base_model = get_peft_model(base_model, lora_cfg)

    if hasattr(base_model, "gradient_checkpointing_enable"):
        try:
            base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            base_model.gradient_checkpointing_enable()
    for mod in base_model.modules():
        if hasattr(mod, "gradient_checkpointing"):
            setattr(mod, "gradient_checkpointing", True)

    model = MultiLayerLoRAClassifier(base_model, hidden_size=base_cfg.hidden_size, layers=layers, num_classes=K)
    # If device_map was used, do NOT call model.to() as it overrides device_map onto a single GPU
    if hasattr(base_model, "hf_device_map") and len(getattr(base_model, "hf_device_map", {})) > 1:
        model.plain_head = model.plain_head.to(head_dev)
        model.centered_head = model.centered_head.to(head_dev)
        for idx in range(len(model.norms)):
            model.norms[idx] = model.norms[idx].to(head_dev)
    else:
        model = model.to(device)

    # Stage 1: Pool Pretraining
    stage1_cfg = train_cfg.get("stage1", {})
    if stage1_cfg.get("enabled", True) and (all_pools or pool_file):
        max_stage1_hours = float(stage1_cfg.get("max_hours", 5.5))
        stage1_start = time.time()
        pool_list = all_pools if all_pools else ([pool_file] if pool_file else [])
        print(f"[Trainer] Stage 1 Pretraining on {len(pool_list)} pool file(s)...")
        dfs = []
        for p in pool_list:
            try:
                if str(p).endswith(".parquet"):
                    df_one = pd.read_parquet(p, columns=["text", "model"])
                else:
                    df_one = pd.read_csv(p, usecols=["text", "model"])
                dfs.append(df_one)
            except Exception as e:
                print(f"[Trainer] Could not read pool {p}: {e}")

        if dfs:
            df_p = pd.concat(dfs, ignore_index=True)
        else:
            df_p = pd.DataFrame(columns=["text", "model"])

        df_p = df_p.dropna(subset=["text", "model"])
        df_p["text"] = df_p["text"].astype(str)
        df_p = df_p[df_p["text"].str.strip().str.len() >= 40]
        df_p = df_p[df_p["model"].isin(LABELS)]

        df_p["_hash"] = [_text_hash(t) for t in df_p["text"]]
        df_p = df_p.drop_duplicates(subset=["_hash"]).reset_index(drop=True)

        # Build unseen holdout to quarantine holdout passages from Stage 1 pretraining
        try:
            _, holdout_hashes = build_unseen_holdout_dataset(input_dir, train_path)
            if holdout_hashes:
                mask_clean = ~df_p["_hash"].isin(holdout_hashes)
                n_dropped = len(df_p) - mask_clean.sum()
                if n_dropped > 0:
                    print(f"[Trainer] Quarantined {n_dropped} pool rows that overlapped with unseen holdout.")
                df_p = df_p[mask_clean].reset_index(drop=True)
        except Exception as e:
            print(f"[Trainer] Note on holdout quarantine: {e}")

        max_s = int(stage1_cfg.get("max_samples", 90000))
        if len(df_p) > max_s:
            target_per_c = max_s // K
            sampled = []
            remainder = []
            for c, group in df_p.groupby("model"):
                if len(group) <= target_per_c:
                    sampled.append(group)
                else:
                    s_grp = group.sample(n=target_per_c, random_state=42)
                    sampled.append(s_grp)
                    remainder.append(group.drop(s_grp.index))
            res_df = pd.concat(sampled, ignore_index=True)
            needed = max_s - len(res_df)
            if needed > 0 and remainder:
                rem_df = pd.concat(remainder, ignore_index=True)
                take_extra = rem_df.sample(n=min(needed, len(rem_df)), random_state=42)
                res_df = pd.concat([res_df, take_extra], ignore_index=True)
            df_p = res_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
        else:
            df_p = df_p.sample(frac=1.0, random_state=42).reset_index(drop=True)

        lengths = [len(tok(t, add_special_tokens=False, truncation=True, max_length=max_len)["input_ids"]) for t in df_p["text"]]
        bs1 = int(stage1_cfg.get("batch_size", 2))
        accum1 = int(stage1_cfg.get("gradient_accumulation_steps", 8))
        ds1 = TextSequenceDataset(df_p["text"].tolist(), [LAB2I[m] for m in df_p["model"]], None, tok, max_len=max_len, end_id=end_id)
        bucket_sampler = BucketedLengthBatchSampler(lengths, batch_size=bs1, shuffle=True)
        loader1 = DataLoader(ds1, batch_sampler=bucket_sampler, collate_fn=collate_text_batch)

        opt1 = torch.optim.AdamW([
            {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": float(stage1_cfg.get("lr_lora", 1.5e-4))},
            {"params": list(model.norms.parameters()) + list(model.plain_head.parameters()), "lr": float(stage1_cfg.get("lr_head", 8e-4))}
        ], weight_decay=float(train_cfg.get("weight_decay", 0.01)))
        sched1 = get_cosine_schedule_with_warmup(opt1, num_warmup_steps=int(stage1_cfg.get("warmup_steps", 100)), num_training_steps=math.ceil(len(loader1) / accum1))

        model.train()
        opt1.zero_grad()
        for step_idx, batch in enumerate(loader1):
            if (step_idx + 1) % 50 == 0:
                elapsed_hours = (time.time() - stage1_start) / 3600.0
                if elapsed_hours >= max_stage1_hours:
                    print(f"[Trainer] Stage 1 reached time budget ({max_stage1_hours}h at step {step_idx + 1}); advancing to Stage 2...")
                    break

            mb_ids = batch["input_ids"].to(device)
            mb_mask = batch["attention_mask"].to(device)
            max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
            mb_ids = mb_ids[:, :max_l]
            mb_mask = mb_mask[:, :max_l]
            labels_dev = batch["labels"].to(device)

            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=torch.float16):
                _, logits_p, _ = model(mb_ids, mb_mask, drop_first_token=not is_deberta)
                loss = family_margin_cross_entropy(logits_p, labels_dev, family_margin=family_margin) / accum1

            loss.backward()
            if (step_idx + 1) % accum1 == 0 or (step_idx + 1) == len(loader1):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt1.step()
                sched1.step()
                opt1.zero_grad()

        del ds1, loader1, opt1, sched1, df_p
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Intermediate checkpoint export after Stage 1
        try:
            export_compatible_checkpoint(
                model=model,
                output_dir=out_dir,
                layers=layers,
                keep=keep_layers,
                max_len=max_len,
                end_id=end_id,
                backbone_name=backbone_name
            )
            print(f"[Trainer] Safe intermediate Stage 1 checkpoint exported to {out_dir}")
        except Exception as e:
            print(f"[Trainer] Warning: could not export Stage 1 intermediate checkpoint: {e}")

    # Stage 2: Prompt-Grouped Fine-Tuning on train.csv
    stage2_cfg = train_cfg.get("stage2", {})
    df_tr = pd.read_csv(train_path)
    print(f"[Trainer] Stage 2 Fine-Tuning on {train_path} ({len(df_tr)} samples)...")
    bs2 = int(stage2_cfg.get("batch_size", 2))
    ds2 = TextSequenceDataset(df_tr["text"].tolist(), [LAB2I[m] for m in df_tr["model"]], df_tr["prompt_id"].tolist(), tok, max_len=max_len, end_id=end_id)
    sampler2 = PromptGroupBatchSampler(df_tr["prompt_id"].tolist(), batch_size=bs2, shuffle=True)
    loader2 = DataLoader(ds2, batch_sampler=sampler2, collate_fn=collate_text_batch)

    opt2 = torch.optim.AdamW([
        {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": float(stage2_cfg.get("lr_lora", 3e-5))},
        {"params": list(model.norms.parameters()) + list(model.plain_head.parameters()) + list(model.centered_head.parameters()), "lr": float(stage2_cfg.get("lr_head", 2e-4))}
    ], weight_decay=float(train_cfg.get("weight_decay", 0.01)))
    epochs2 = int(stage2_cfg.get("epochs", 2))
    total_steps2 = len(loader2) * epochs2
    sched2 = get_cosine_schedule_with_warmup(opt2, num_warmup_steps=int(stage2_cfg.get("warmup_steps", 50)), num_training_steps=total_steps2)

    model.train()
    for ep in range(epochs2):
        sampler2.set_epoch(ep)
        opt2.zero_grad()
        for step_idx, batch in enumerate(loader2):
            train_prompt_group_step(
                model=model,
                opt=opt2,
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["labels"].to(device),
                micro_batch_size=bs2,
                centered_weight=float(stage2_cfg.get("centered_weight", 0.60)),
                family_margin=family_margin,
                supcon_weight=supcon_weight,
                label_smoothing=0.10
            )
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt2.step()
            sched2.step()
            opt2.zero_grad()

    # Feature extraction for wPCA bank & checkpoint export
    del loader2, opt2, sched2, sampler2
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model.eval()
    loader_eval = DataLoader(ds2, batch_size=4, shuffle=False, collate_fn=collate_text_batch)
    Z_list = []
    with torch.no_grad():
        for batch in loader_eval:
            z_b = model.forward_features(batch["input_ids"].to(device), batch["attention_mask"].to(device), drop_first_token=not is_deberta)
            Z_list.append(z_b.cpu().numpy())
    Z_train = np.vstack(Z_list)
    y_train = np.array([LAB2I[m] for m in df_tr["model"]])
    prompts_train = df_tr["prompt_id"].values

    export_compatible_checkpoint(
        model=model,
        output_dir=out_dir,
        layers=layers,
        keep=keep_layers,
        max_len=max_len,
        end_id=end_id,
        backbone_name=backbone_name,
        Z_train=Z_train,
        y_train=y_train,
        prompts_train=prompts_train
    )
    print(f"[Trainer] Checkpoint exported to {out_dir}")
    return out_dir


In [ ]:
if __name__ == '__main__':
    run_training_pipeline(TRAINING_CONFIG)
